# NWS forecast archive: AWS NDFD

This spike asks whether archived official NWS forecasts reproduce the forecasts collected
for the seven configured stations, and whether they can support historical Baseline 1 inputs.
It runs on the Mac. Nothing in `src/` imports this notebook. No private bucket or VM is used.

**Approval incorporated:** AWS only; KNYC first; approximately 30 historical AWS days plus
every copied overlap day (2026-11-07 through 2026-11-13). Sample the NDFD cell containing the
centroid of the **collected forecastGridData polygon**, never the API gridX/gridY indices.
The latter establish API endpoint identity only. No full backfill is run.

The requested availability inequality was clarified interactively: keep
`L = maximum observed overlap lag + 30 minutes`, and require reconstructed availability
**at or after** the first matching collected fetch. The originally requested reverse
inequality is incompatible with that formula. Two separate decision lines are drafted below;
neither is written to `docs/decisions.md`.

**Read results in order:** raw-input accounting → archive inventory → location/GRIB decoding →
overlap controls → lag rule and deliberate failures → historical format/coverage survey →
four-decision versus every-issuance costs → full mismatch/lag tables and drafted findings.
If the exact overlap control fails, historical values remain **unvalidated**; the survey
only establishes archive/format coverage. No fallback candidate is accessed automatically.

## Verified documentation and reproducibility

- [NOAA AWS registry](https://registry.opendata.aws/noaa-ndfd/) documents the public,
  unauthenticated `noaa-ndfd-pds` bucket, the layout
  `wmo/<parameter>/YYYY/MM/DD/<WMO-filename>`, updates as often as half-hourly, and recommends
  `grib2io` because other decoders can mishandle NDFD scanning conventions. The
  [NOAA product mapping](https://noaa-ndfd-pds.s3.amazonaws.com/NDFDelem_fullres_202206.xls)
  identifies `maxt`, operational 2.5 km CONUS `YGUZ98` (days 1–3) and `YGUZ97` (days 4–7),
  both `KWBN`, distributed as `ds.maxt.bin`. `YGUZ87/88` are legacy 5 km products and excluded.
  Native MaxT units are Fahrenheit; GRIB uses Kelvin with one decimal place.
- [NCEI's NDFD product documentation](https://www.ncei.noaa.gov/products/weather-climate-models/national-digital-forecast-database)
  documents AWS coverage from **2020-04-16**. This was read during plan research; this
  implementation accesses no NCEI or MOS archive. Directory existence, CONUS file inventories,
  and decoded point coverage are reported separately rather than equated.
- Public [ListObjectsV2](https://docs.aws.amazon.com/AmazonS3/latest/API/API_ListObjectsV2.html)
  requests use `list-type=2`, `prefix`, optional `delimiter`, `max-keys`, and continuation
  tokens until `IsTruncated=false`. Object GETs follow
  [GetObject](https://docs.aws.amazon.com/AmazonS3/latest/API/API_GetObject.html).
  Every response body is cached unchanged, with status, headers, hash, request/receipt times,
  and measured transfer time. S3 Last-Modified is audit metadata **never availability**.
- [NWS API grid documentation](https://weather-gov.github.io/api/gridpoints) specifies
  `maxTemperature.uom`, `values[].value`, ISO `validTime` intervals, nulls, and merged equal
  values. API `updateTime` is grid creation time, not publication time.
- [NWSI 10-201](https://www.weather.gov/media/directives/010_pdfs/pd01002001curr.pdf)
  describes WFO-derived NDFD mosaics and defines daytime MaxT as **07:00–19:00 local standard
  time**. Shared lineage does not guarantee every WFO/API revision is retained. Encoded
  CONUS UTC interval labels and API value-support intervals are preserved separately;
  this notebook does not settle the CLI climate-day question.
- [grib2io API](https://noaa-mdl.github.io/grib2io/grib2io.html),
  [NOAA g2c build documentation](https://github.com/NOAA-EMC/NCEPLIBS-g2c/tree/v2.3.0),
  [GRIB time units](https://www.nco.ncep.noaa.gov/pmb/docs/grib2/grib2_doc/grib2_table4-4.shtml),
  and [maximum statistic](https://www.nco.ncep.noaa.gov/pmb/docs/grib2/grib2_doc/grib2_table4-10.shtml)
  define the decoding used here. PDT 4.8's explicit end timestamp is authoritative. During
  development the valid-period check caught the erroneous shortcut `reference + leadTime`
  on already-started intervals; the corrected decoder is run over the original cached files.

Requests are serial with starts at least one second apart, bounded retries, and Retry-After
handling. No numeric NOAA archive quota was found; this is a conservative client policy.
Cached listings are frozen snapshots. A later copied collection window needs newly inspected
listing coverage; an old partial-day snapshot must not be mistaken for a complete new day.

Dependencies `grib2io` and `pyproj` are in the **notebooks** uv group. Runtime dependencies
are unchanged. Native g2c 2.3.0 was built locally below the ignored spike directory, with
unused PNG/JPEG/interpolation features disabled; all sampled files use supported complex
packing. No global Homebrew changes were made. To resync this Mac's notebook environment:

```sh
CMAKE_PREFIX_PATH="$PWD/data/spike/nws_archive/native/install" uv sync --group notebooks
```

The cached source is under `data/spike/nws_archive/native/source/`. If rebuilding it, use
NOAA's documented CMake commands with `-DCMAKE_INSTALL_PREFIX` pointing at the cache's
`native/install`, `-DUSE_PNG=OFF -DUSE_Jasper=OFF -DUSE_OpenJPEG=OFF -DBUILD_TESTING=OFF -DUTILS=OFF`.
`uv tool run --from cmake cmake` provides CMake without a runtime project dependency.
The project records `GRIB2IO_WITH_IP=OFF`: point selection requires no interpolation library.
On a fresh machine, install/build native g2c before syncing the notebooks group.

Start the notebook from the repo root or `notebooks/`. All raw and derived spike artifacts
live under `data/spike/nws_archive/`, verified by `git check-ignore` in the first cell.
Rerunning an identical request reads its cached response without contacting the API.

## Original copied inputs, station identity, and daily accounting

The reader handles loose `.json.gz`, regular hourly `.tar.zst`, and `.late-N.tar.zst`.
It normalizes each member back to its original filename, compares decompressed envelope
bytes exactly in a disk-backed index, and fails on a conflicting duplicate. Raw API bodies
remain unchanged. Format counts are appearances; duplicates plus unique responses reconcile
to those appearances. `received_at` is our fetch time, never an issue timestamp.

In [1]:
import base64
import csv
import gzip
import hashlib
import io
import json
import re
import sqlite3
import subprocess
import tarfile
from collections import Counter, defaultdict
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import pandas as pd
import zstandard
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config/nws_stations.csv").exists():
    ROOT = ROOT.parent
CACHE = ROOT / "data/spike/nws_archive"
COLLECTED = CACHE / "collected/nws"
assert (
    subprocess.run(
        ["git", "check-ignore", "data/spike/nws_archive/"], cwd=ROOT, capture_output=True
    ).returncode
    == 0
), "The exact spike cache path must be gitignored."
assert COLLECTED.is_dir(), "Copy the original NWS tree to the documented collected path."
with (ROOT / "config/nws_stations.csv").open() as config_file:
    STATIONS = list(csv.DictReader(config_file))
assert len(STATIONS) == 7


def utc(value):
    result = datetime.fromisoformat(value.replace("Z", "+00:00"))
    assert result.utcoffset() == timedelta(0), f"Expected aware UTC: {value}"
    return result


LOOSE = re.compile(r"(\d{4}-\d{2}-\d{2})T\d{6}\.\d{6}Z\.json\.gz")
BUNDLE = re.compile(r"(\d{4}-\d{2}-\d{2})T\d{2}(\.late-[1-9]\d*)?\.tar\.zst")


def input_payloads(input_root):
    for source_file in sorted(input_root.rglob("*")):
        if not source_file.is_file():
            continue
        loose_match = LOOSE.fullmatch(source_file.name)
        bundle_match = BUNDLE.fullmatch(source_file.name)
        if loose_match:
            assert source_file.parent.name == loose_match[1], "Loose directory date differs."
            yield (
                loose_match[1],
                "loose",
                source_file.name,
                gzip.decompress(source_file.read_bytes()),
                str(source_file.relative_to(input_root)),
            )
        elif bundle_match:
            assert source_file.parent.name == bundle_match[1], "Archive directory date differs."
            kind = "late" if bundle_match[2] else "compacted"
            with (
                source_file.open("rb") as raw_stream,
                zstandard.ZstdDecompressor().stream_reader(raw_stream) as decoded_stream,
                tarfile.open(fileobj=decoded_stream, mode="r|") as bundle,
            ):
                for member in bundle:
                    original_name = member.name + ".gz"
                    assert member.isfile() and LOOSE.fullmatch(original_name), (
                        f"Unexpected archive member: {source_file}:{member.name}"
                    )
                    assert original_name[:13] == source_file.name[:13], "Member hour differs."
                    with bundle.extractfile(member) as member_stream:
                        payload = member_stream.read()
                    assert len(payload) == member.size, "Truncated archive member."
                    yield (
                        bundle_match[1],
                        kind,
                        original_name,
                        payload,
                        (str(source_file.relative_to(input_root)) + ":" + member.name),
                    )
                    bundle.members.clear()
                while decoded_stream.read(64 * 1024):
                    pass
        else:
            assert source_file.name.startswith(".compact"), f"Unexpected input: {source_file}"


def read_inputs(input_root, database):
    # Disk-backed exact-byte comparison bounds memory even after future larger copies.
    connection = sqlite3.connect(database)
    connection.execute("DROP TABLE IF EXISTS responses")
    connection.execute("CREATE TABLE responses(name TEXT PRIMARY KEY, payload BLOB)")
    counts = defaultdict(Counter)
    parsed = []
    for day, kind, original_name, payload, provenance in input_payloads(input_root):
        counts[day][kind] += 1
        previous = connection.execute(
            "SELECT payload FROM responses WHERE name=?", (original_name,)
        ).fetchone()
        if previous is not None:
            assert zstandard.decompress(previous[0]) == payload, (
                f"Different bodies share original name {original_name}"
            )
            counts[day]["duplicates"] += 1
            continue
        connection.execute(
            "INSERT INTO responses VALUES (?, ?)",
            (original_name, zstandard.compress(payload, level=1)),
        )
        envelope = json.loads(payload)
        received = utc(envelope["received_at"])
        assert received.date().isoformat() == day, "Envelope and file dates disagree."
        assert received.strftime("%Y-%m-%dT%H%M%S.%fZ.json.gz") == original_name
        assert utc(envelope["requested_at"]) <= received
        encoding = envelope.get("body_encoding")
        assert encoding in (None, "base64"), f"Unknown body encoding: {encoding}"
        body = (
            envelope["body"].encode("utf-8")
            if encoding is None
            else base64.b64decode(envelope["body"], validate=True)
        )
        counts[day]["unique"] += 1
        counts[day][f"http_{envelope['http_status']}"] += 1
        url = envelope["request_url"]
        record = dict(
            name=original_name,
            url=url,
            fetched_at=received,
            status=envelope["http_status"],
            provenance=provenance,
            body_sha256=hashlib.sha256(body).hexdigest(),
        )
        if record["status"] == 200 and (
            "/points/" in url
            or re.fullmatch(r"https://api.weather.gov/gridpoints/[A-Z]+/\d+,\d+", url)
        ):
            parsed_body = json.loads(body)
            selected = (
                ("forecastGridData", "gridId", "gridX", "gridY", "timeZone")
                if "/points/" in url
                else ("maxTemperature", "updateTime", "validTimes")
            )
            record["data"] = {
                "geometry": parsed_body["geometry"],
                "properties": {k: parsed_body["properties"][k] for k in selected},
            }
            counts[day]["points" if "/points/" in url else "forecastGridData"] += 1
        parsed.append(record)
    connection.commit()
    connection.execute("VACUUM")
    connection.close()
    rows = []
    for day, tally in sorted(counts.items()):
        assert sum(tally[k] for k in ("loose", "compacted", "late")) == (
            tally["unique"] + tally["duplicates"]
        ), "Input accounting does not reconcile."
        rows.append(
            {
                "day": day,
                **{
                    k: tally[k]
                    for k in (
                        "loose",
                        "compacted",
                        "late",
                        "duplicates",
                        "unique",
                        "points",
                        "forecastGridData",
                    )
                },
                "HTTP": {k: v for k, v in tally.items() if k.startswith("http_")},
            }
        )
    return parsed, pd.DataFrame(rows)


RESPONSES, INPUT_COUNTS = read_inputs(COLLECTED, CACHE / "input_comparison.sqlite")
display(INPUT_COUNTS)

# /points establishes station identity. The grid polygon establishes NDFD sampling location.
POINTS = defaultdict(list)
for station in STATIONS:
    station_id = station["station_id"]
    points_url = (
        f"https://api.weather.gov/points/{float(station['latitude']):.4f},"
        f"{float(station['longitude']):.4f}"
    )
    POINTS[station_id] = sorted(
        [r for r in RESPONSES if r["url"] == points_url and "data" in r],
        key=lambda r: r["fetched_at"],
    )
    assert POINTS[station_id], f"No collected /points for {station_id}"

GRIDS = defaultdict(list)
for station_id, points_records in POINTS.items():
    known_urls = {r["data"]["properties"]["forecastGridData"] for r in points_records}
    for record in RESPONSES:
        if record["url"] in known_urls and "data" in record:
            earlier_points = [p for p in points_records if p["fetched_at"] <= record["fetched_at"]]
            assert earlier_points, "A grid fetch precedes every collected /points response."
            mapping = earlier_points[-1]
            assert mapping["data"]["properties"]["forecastGridData"] == record["url"]
            assert record["data"]["geometry"]["type"] == "Polygon"
            GRIDS[station_id].append(record)
    GRIDS[station_id].sort(key=lambda r: r["fetched_at"])
    assert GRIDS[station_id], f"No collected grid response for {station_id}"

display(
    pd.DataFrame(
        [
            dict(
                station=s,
                responses=len(rs),
                grid_issuances=len({r["data"]["properties"]["updateTime"] for r in rs}),
                maximum_vectors=len(
                    {
                        json.dumps(r["data"]["properties"]["maxTemperature"], sort_keys=True)
                        for r in rs
                    }
                ),
                first_fetch=rs[0]["fetched_at"],
                last_fetch=rs[-1]["fetched_at"],
                polygons=len({json.dumps(r["data"]["geometry"], sort_keys=True) for r in rs}),
            )
            for s, rs in GRIDS.items()
        ]
    )
)

,day,loose,compacted,late,duplicates,unique,points,forecastGridData,HTTP
0,2026-11-07,112,0,0,0,112,7,35,{'http_200': 112}
1,2026-11-08,1015,0,0,0,1015,7,336,{'http_200': 1015}
2,2026-11-09,1015,0,0,0,1015,7,336,{'http_200': 1015}
3,2026-11-10,1015,0,0,0,1015,7,336,{'http_200': 1015}
4,2026-11-11,1043,0,0,0,1043,14,343,{'http_200': 1043}
5,2026-11-12,1043,0,0,0,1043,14,343,{'http_200': 1043}
6,2026-11-13,63,0,0,0,63,0,21,{'http_200': 63}


,station,responses,grid_issuances,maximum_vectors,first_fetch,last_fetch,polygons
0,KAUS,250,83,56,2026-11-07 21:45:31.345448+00:00,2026-11-13 01:13:38.121318+00:00,1
1,KMDW,250,60,44,2026-11-07 21:45:35.298741+00:00,2026-11-13 01:13:40.584062+00:00,1
2,KDEN,250,11,11,2026-11-07 21:45:39.405253+00:00,2026-11-13 01:13:43.641330+00:00,1
3,KLAX,250,31,25,2026-11-07 21:45:43.365424+00:00,2026-11-13 01:13:46.852719+00:00,1
4,KMIA,250,34,25,2026-11-07 21:45:47.356033+00:00,2026-11-13 01:13:49.664740+00:00,1
5,KNYC,250,26,21,2026-11-07 21:45:51.305470+00:00,2026-11-13 01:13:52.643025+00:00,1
6,KPHL,250,29,24,2026-11-07 21:45:55.317440+00:00,2026-11-13 01:13:55.748765+00:00,1


## AWS MaxT inventory and exact cell selection

Both operational forecast-range bundles are needed. Archive directory dates and WMO issue
dates are kept separately: the real 2023-01-05 prefix contains an issuance named
`YGUZ98_KWBN_202301042355`. Prefix dates never supply reconstructed issue times.

The containing cell is found in the GRIB projection from the polygon centroid and half-cell
boundaries. The scan flags are asserted. For dates before collection began, the earliest
collected polygon supplies the explicitly documented fixed geographic sampling location;
this does not claim historical API office/grid identity.

Decode one approximately three-million-cell field at a time: about 11.3 MiB for float32,
plus decoder and Python overhead. Release each field before the next. Measured process peak
RSS and timing, including the notebook reader, are reported later.

In [2]:
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from email.utils import parsedate_to_datetime

HTTP_CACHE = CACHE / "http"
HTTP_CACHE.mkdir(exist_ok=True)
REQUEST_LOG = []
NEXT_REQUEST = 0.0
AWS = "https://noaa-ndfd-pds.s3.amazonaws.com/"
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}


def cached_get(url):
    global NEXT_REQUEST
    identity = hashlib.sha256(url.encode()).hexdigest()
    body_file = HTTP_CACHE / (identity + ".body")
    meta_file = HTTP_CACHE / (identity + ".json")
    if meta_file.exists():
        metadata = json.loads(meta_file.read_text())
        assert metadata["url"] == url
        body = body_file.read_bytes()
        assert hashlib.sha256(body).hexdigest() == metadata["sha256"]
        assert metadata["status"] == 200, f"Cached HTTP error: {metadata}"
        REQUEST_LOG.append({**metadata, "cached": True})
        return body, metadata
    for attempt in range(4):
        time.sleep(max(0, NEXT_REQUEST - time.monotonic()))
        NEXT_REQUEST = time.monotonic() + 1.0
        started = datetime.now(UTC)
        monotonic_start = time.monotonic()
        request = urllib.request.Request(
            url,
            headers={
                "User-Agent": (
                    "Kalshi-Weather-Bot/archive-spike (github.com/prathikn20/Kalshi-Weather-Bot)"
                ),
                "Accept-Encoding": "identity",
            },
        )
        try:
            try:
                response = urllib.request.urlopen(request, timeout=60)
            except urllib.error.HTTPError as error:
                response = error
            with response:
                body = response.read()
                metadata = dict(
                    url=url,
                    status=response.status,
                    headers=list(response.headers.items()),
                    requested_at=started.isoformat(),
                    received_at=datetime.now(UTC).isoformat(),
                    seconds=time.monotonic() - monotonic_start,
                    size=len(body),
                    sha256=hashlib.sha256(body).hexdigest(),
                    attempt=attempt + 1,
                )
        except (OSError, TimeoutError):
            if attempt == 3:
                raise
            time.sleep(2**attempt)
            continue
        # Every HTTP response is preserved, including unsuccessful retry attempts.
        suffix = "" if metadata["status"] == 200 else f".attempt-{attempt}-{started.timestamp()}"
        (HTTP_CACHE / (identity + suffix + ".body")).write_bytes(body)
        (HTTP_CACHE / (identity + suffix + ".json")).write_text(json.dumps(metadata, indent=2))
        REQUEST_LOG.append({**metadata, "cached": False})
        if metadata["status"] == 200:
            return body, metadata
        if metadata["status"] not in (408, 429, 500, 502, 503, 504) or attempt == 3:
            raise RuntimeError(f"HTTP {metadata['status']}: {url}")
        retry_after = dict(metadata["headers"]).get("Retry-After", "0")
        try:
            retry_delay = float(retry_after)
        except ValueError:
            retry_delay = (parsedate_to_datetime(retry_after) - datetime.now(UTC)).total_seconds()
        time.sleep(max(2**attempt, retry_delay))
    raise AssertionError("Unreachable retry state")


def list_objects(prefix, delimiter=None):
    objects, prefixes, tokens = [], [], set()
    continuation = None
    while True:
        params = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if delimiter:
            params["delimiter"] = delimiter
        if continuation:
            params["continuation-token"] = continuation
        payload, _ = cached_get(AWS + "?" + urllib.parse.urlencode(params))
        document = ET.fromstring(payload)
        assert document.findtext("s3:Prefix", namespaces=NS) == prefix
        for obj in document.findall("s3:Contents", NS):
            objects.append(
                dict(
                    key=obj.findtext("s3:Key", namespaces=NS),
                    size=int(obj.findtext("s3:Size", namespaces=NS)),
                    # Audit metadata only. NEVER used to reconstruct availability.
                    s3_last_modified=obj.findtext("s3:LastModified", namespaces=NS),
                    etag=obj.findtext("s3:ETag", namespaces=NS),
                )
            )
        prefixes.extend(
            p.findtext("s3:Prefix", namespaces=NS)
            for p in document.findall("s3:CommonPrefixes", NS)
        )
        if document.findtext("s3:IsTruncated", namespaces=NS) == "false":
            break
        continuation = document.findtext("s3:NextContinuationToken", namespaces=NS)
        assert continuation and continuation not in tokens, "Missing or looping pagination token."
        tokens.add(continuation)
    assert len({obj["key"] for obj in objects}) == len(objects), "Duplicate object across pages."
    return objects, prefixes


def day_inventory(day):
    # U is the CONUS full-resolution sector; both projection ranges are needed.
    objects, _ = list_objects(f"wmo/maxt/{day:%Y/%m/%d}/YGU")
    for obj in objects:
        match = re.fullmatch(r"YGUZ(87|88|97|98)_KWBN_(\d{12})", obj["key"].split("/")[-1])
        assert match, f"Unexpected CONUS MaxT product: {obj['key']}"
        obj["projection"] = match[1]
        obj["filename_issue"] = datetime.strptime(match[2], "%Y%m%d%H%M").replace(tzinfo=UTC)
        # Prefix day is an archive directory, not a guaranteed issue date: the real
        # 2023-01-05 directory includes YGUZ98_KWBN_202301042355. Keep both explicitly.
        obj["directory_day"] = day.isoformat()
        obj["issue_date_matches_directory"] = obj["filename_issue"].date() == day
    # The official spreadsheet identifies 87/88 as legacy 5 km, 97/98 as operational 2.5 km.
    return [obj for obj in objects if obj["projection"] in ("97", "98")]


# First verify actual product-family presence. No NCEI or MOS requests are made.
OVERLAP_DAYS = sorted({r["fetched_at"].date() for r in GRIDS["KNYC"]})
# Include the preceding day so an issuance first seen after the copied window begins is represented.
OVERLAP_INVENTORY_DAYS = [OVERLAP_DAYS[0] - timedelta(days=1), *OVERLAP_DAYS]
OVERLAP_OBJECTS = [obj for d in OVERLAP_INVENTORY_DAYS for obj in day_inventory(d)]
display(
    pd.DataFrame(OVERLAP_OBJECTS)
    .assign(day=lambda frame: frame.filename_issue.dt.date)
    .groupby(["day", "projection"])
    .agg(
        objects=("key", "size"),
        bytes=("size", "sum"),
        first_issue=("filename_issue", "min"),
        last_issue=("filename_issue", "max"),
    )
)

objects      bytes               first_issue  \
day        projection                                                 
2026-11-06 97                5   20741234 2026-11-06 05:33:00+00:00   
           98               48  154788050 2026-11-06 00:16:00+00:00   
2026-11-07 97                5   20774659 2026-11-07 05:34:00+00:00   
           98               48  154147329 2026-11-07 00:16:00+00:00   
2026-11-08 97                5   20766104 2026-11-08 05:34:00+00:00   
           98               48  153702435 2026-11-08 00:15:00+00:00   
2026-11-09 97                5   20791699 2026-11-09 05:34:00+00:00   
           98               47  150869834 2026-11-09 00:16:00+00:00   
2026-11-10 97                5   20799546 2026-11-10 05:34:00+00:00   
           98               48  154819014 2026-11-10 00:15:00+00:00   
2026-11-11 97                5   20817047 2026-11-11 05:33:00+00:00   
           98               48  154119420 2026-11-11 00:16:00+00:00   
2026-11-12 97                5   20119760 2026-11-12 05:34:00+00:00   
           98               48  153693115 2026-11-12 00:16:00+00:00   
2026-11-13 98                6   18677078 2026-11-13 00:16:00+00:00   

                                     last_issue  
day        projection                            
2026-11-06 97         2026-11-06 23:33:00+00:00  
           98         2026-11-06 23:46:00+00:00  
2026-11-07 97         2026-11-07 23:33:00+00:00  
           98         2026-11-07 23:46:00+00:00  
2026-11-08 97         2026-11-08 23:33:00+00:00  
           98         2026-11-08 23:46:00+00:00  
2026-11-09 97         2026-11-09 23:33:00+00:00  
           98         2026-11-09 23:46:00+00:00  
2026-11-10 97         2026-11-10 23:33:00+00:00  
           98         2026-11-10 23:47:00+00:00  
2026-11-11 97         2026-11-11 23:33:00+00:00  
           98         2026-11-11 23:47:00+00:00  
2026-11-12 97         2026-11-12 23:33:00+00:00  
           98         2026-11-12 23:46:00+00:00  
2026-11-13 98         2026-11-13 02:46:00+00:00

In [3]:
import math
import resource

import grib2io
import numpy as np
from pyproj import Proj

GRIB_CACHE = CACHE / "grib"
GRIB_CACHE.mkdir(exist_ok=True)
DECODE_LOG = []
GRID_LOCATIONS = {}
REFERENCE_DIFFERENCES = []


def polygon_centroid(geometry):
    assert geometry["type"] == "Polygon" and len(geometry["coordinates"]) == 1
    vertices = np.array(geometry["coordinates"][0], dtype=float)
    assert np.array_equal(vertices[0], vertices[-1]), "Unclosed API grid polygon."
    # Translation improves numerical conditioning of the shoelace formula.
    origin = vertices[0]
    vertices = vertices - origin
    cross = vertices[:-1, 0] * vertices[1:, 1] - vertices[1:, 0] * vertices[:-1, 1]
    area_twice = cross.sum()
    assert abs(area_twice) > 1e-12, "Degenerate grid polygon."
    centroid = ((vertices[:-1] + vertices[1:]) * cross[:, None]).sum(axis=0) / (3 * area_twice)
    return tuple(centroid + origin)


def containing_cell(message, geometry):
    assert int(message.gdtn) == 30, "Expected CONUS Lambert conformal grid."
    assert message.scanModeFlags[:4] == [0, 1, 0, 1], "Unexpected NDFD scan flags."
    # grib2io reverses alternate rows in unpacked data. Coordinates increase E and N.
    projection = Proj(message.projParameters)
    x0, y0 = projection(message.longitudeFirstGridpoint, message.latitudeFirstGridpoint)
    dx, dy = message.gridlengthXDirection, message.gridlengthYDirection
    longitude, latitude = polygon_centroid(geometry)
    x, y = projection(longitude, latitude)
    fractional_x, fractional_y = (x - x0) / dx, (y - y0) / dy
    column, row = math.floor(fractional_x + 0.5), math.floor(fractional_y + 0.5)
    validate_cell(row, column, fractional_y, fractional_x, message.ny, message.nx)
    center_lon, center_lat = projection(x0 + column * dx, y0 + row * dy, inverse=True)
    return dict(
        row=row,
        column=column,
        fractional_x=fractional_x,
        fractional_y=fractional_y,
        centroid_lon=longitude,
        centroid_lat=latitude,
        center_lon=center_lon,
        center_lat=center_lat,
        dx=dx,
        dy=dy,
        nx=int(message.nx),
        ny=int(message.ny),
        scan_flags=message.scanModeFlags,
    )


def validate_cell(row, column, fractional_y, fractional_x, ny, nx):
    assert 0 <= row < ny and 0 <= column < nx, "Centroid is outside the NDFD domain."
    assert abs(row - fractional_y) <= 0.5 and abs(column - fractional_x) <= 0.5, (
        "Selected NDFD cell does not contain the collected API polygon centroid."
    )


def decode_object(obj, station_ids=("KNYC",)):
    raw, metadata = cached_get(AWS + obj["key"])
    assert len(raw) == obj["size"], "Object size differs from its listing."
    # The exact HTTP body remains in http/. This named link makes GRIB inspection convenient.
    grib_path = GRIB_CACHE / obj["key"].split("/")[-1]
    if not grib_path.exists():
        grib_path.hardlink_to(
            HTTP_CACHE / (hashlib.sha256((AWS + obj["key"]).encode()).hexdigest() + ".body")
        )
    else:
        assert grib_path.read_bytes() == raw, "Named GRIB cache differs from raw response."
    started = time.monotonic()
    rows = []
    with grib2io.open(str(grib_path), save_index=False, use_index=False) as messages:
        for message in messages:
            assert message.shortName == "TMAX" and message.units == "K"
            assert int(message.statisticalProcess) == 2, "Expected maximum statistic."
            assert int(message.pdtn) == 8, "Expected statistical interval template 4.8."
            reference = message.refDate.replace(tzinfo=UTC)
            issue = obj[
                "filename_issue"
            ]  # Explicit choice: WMO filename, not nominal GRIB reference.
            REFERENCE_DIFFERENCES.append(
                dict(
                    key=obj["key"],
                    filename_issue=issue,
                    grib_reference=reference,
                    difference_minutes=(reference - issue).total_seconds() / 60,
                )
            )
            duration = message.duration
            # PDT 4.8 explicitly encodes interval end. The convenience leadTime can be
            # clamped for an already-started interval; reference+leadTime is then wrong.
            end = datetime(
                message.yearOfEndOfTimePeriod,
                message.monthOfEndOfTimePeriod,
                message.dayOfEndOfTimePeriod,
                message.hourOfEndOfTimePeriod,
                message.minuteOfEndOfTimePeriod,
                message.secondOfEndOfTimePeriod,
                tzinfo=UTC,
            )
            assert end == message.validDate.replace(tzinfo=UTC)
            start = end - duration
            assert duration == timedelta(hours=12), f"Unexpected MaxT duration: {duration}"
            assert end > start
            field = message.data
            assert field.shape == (message.ny, message.nx)
            for station_id in station_ids:
                # Select collected polygon as of issue when available. Earlier history uses the
                # earliest collected polygon explicitly, without claiming historical WFO identity.
                eligible = [r for r in GRIDS[station_id] if r["fetched_at"] <= issue]
                grid_record = eligible[-1] if eligible else GRIDS[station_id][0]
                geometry = grid_record["data"]["geometry"]
                signature = hashlib.sha256(
                    message.section3.tobytes() + json.dumps(geometry, sort_keys=True).encode()
                ).hexdigest()
                if signature not in GRID_LOCATIONS:
                    GRID_LOCATIONS[signature] = containing_cell(message, geometry)
                location = GRID_LOCATIONS[signature]
                kelvin = float(field[location["row"], location["column"]])
                assert math.isfinite(kelvin), f"Missing point value: {obj['key']} {station_id}"
                rows.append(
                    dict(
                        station=station_id,
                        key=obj["key"],
                        issue=issue,
                        grib_reference=reference,
                        valid_start=start,
                        valid_end=end,
                        kelvin=kelvin,
                        celsius=kelvin - 273.15,
                        lead_time_would_mislabel=reference + message.leadTime != end,
                        packing_quantum_k=2.0**message.binScaleFactor
                        * 10.0 ** (-message.decScaleFactor),
                        fahrenheit=(kelvin - 273.15) * 1.8 + 32,
                        polygon_response=grid_record["name"],
                        mapping_extrapolated=not bool(eligible),
                        row=location["row"],
                        column=location["column"],
                    )
                )
            del field
            message.flush_data()
    DECODE_LOG.append(
        dict(
            key=obj["key"],
            seconds=time.monotonic() - started,
            bytes=len(raw),
            peak_rss_mib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2,
        )
    )
    return rows


# One real file establishes location, packing and reference-time behavior before bulk work.
PROBE_OBJECT = next(o for o in OVERLAP_OBJECTS if o["projection"] == "98")
PROBE_ROWS = decode_object(PROBE_OBJECT, tuple(GRIDS))
display(pd.DataFrame(PROBE_ROWS))
display(pd.DataFrame(GRID_LOCATIONS.values()))
display(pd.DataFrame(REFERENCE_DIFFERENCES).drop_duplicates())

,station,key,issue,grib_reference,valid_start,valid_end,kelvin,celsius,lead_time_would_mislabel,packing_quantum_k,fahrenheit,polygon_response,mapping_extrapolated,row,column
0,KAUS,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-06 12:00:00+00:00,2026-11-07 00:00:00+00:00,297.600006,24.450006,False,0.1,76.010011,2026-11-07T214531.345448Z.json.gz,True,332,986
1,KMDW,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-06 12:00:00+00:00,2026-11-07 00:00:00+00:00,282.000000,8.850000,False,0.1,47.930000,2026-11-07T214535.298741Z.json.gz,True,857,1336
2,KDEN,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-06 12:00:00+00:00,2026-11-07 00:00:00+00:00,286.500000,13.350000,False,0.1,56.030000,2026-11-07T214539.405253Z.json.gz,True,774,752
3,KLAX,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-06 12:00:00+00:00,2026-11-07 00:00:00+00:00,300.399994,27.249994,False,0.1,81.049989,2026-11-07T214543.365424Z.json.gz,True,571,232
4,KMIA,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-06 12:00:00+00:00,2026-11-07 00:00:00+00:00,303.100006,29.950006,False,0.1,85.910011,2026-11-07T214547.356033Z.json.gz,True,170,1666
5,KNYC,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-06 12:00:00+00:00,2026-11-07 00:00:00+00:00,292.600006,19.450006,False,0.1,67.010011,2026-11-07T214551.305470Z.json.gz,True,860,1811
6,KPHL,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-06 12:00:00+00:00,2026-11-07 00:00:00+00:00,293.700012,20.550012,False,0.1,68.990022,2026-11-07T214555.317440Z.json.gz,True,813,1774
7,KAUS,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-07 12:00:00+00:00,2026-11-08 00:00:00+00:00,295.399994,22.249994,False,0.1,72.049989,2026-11-07T214531.345448Z.json.gz,True,332,986
8,KMDW,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-07 12:00:00+00:00,2026-11-08 00:00:00+00:00,284.300018,11.150018,False,0.1,52.070033,2026-11-07T214535.298741Z.json.gz,True,857,1336
9,KDEN,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,2026-11-07 12:00:00+00:00,2026-11-08 00:00:00+00:00,288.700012,15.550012,False,0.1,59.990022,2026-11-07T214539.405253Z.json.gz,True,774,752


,row,column,fractional_x,fractional_y,centroid_lon,centroid_lat,center_lon,center_lat,dx,dy,nx,ny,scan_flags
0,332,986,985.992301,332.004629,-97.684210,30.180325,-97.684005,30.180223,2539.703,2539.703,2145,1377,"[0, 1, 0, 1, 0, 0, 0, 0]"
1,857,1336,1335.992676,857.013508,-87.740176,41.794394,-87.739983,41.794091,2539.703,2539.703,2145,1377,"[0, 1, 0, 1, 0, 0, 0, 0]"
2,774,752,751.995827,774.011220,-104.656039,39.857226,-104.655897,39.856986,2539.703,2539.703,2145,1377,"[0, 1, 0, 1, 0, 0, 0, 0]"
3,571,232,231.994762,571.008714,-118.388250,33.941450,-118.388069,33.941277,2539.703,2539.703,2145,1377,"[0, 1, 0, 1, 0, 0, 0, 0]"
4,170,1666,1665.993123,170.002011,-80.310841,25.794623,-80.310673,25.794561,2539.703,2539.703,2145,1377,"[0, 1, 0, 1, 0, 0, 0, 0]"
5,860,1811,1810.992115,860.012884,-73.971000,40.776150,-73.970832,40.775844,2539.703,2539.703,2145,1377,"[0, 1, 0, 1, 0, 0, 0, 0]"
6,813,1774,1773.991181,813.013215,-75.229400,39.876300,-75.229205,39.875984,2539.703,2539.703,2145,1377,"[0, 1, 0, 1, 0, 0, 0, 0]"


,key,filename_issue,grib_reference,difference_minutes
0,wmo/maxt/2026/11/06/YGUZ98_KWBN_202611060016,2026-11-06 00:16:00+00:00,2026-11-06 00:30:00+00:00,14.0


## Overlap: value search first, strict period control second

The next cell is a **diagnostic search**, not the final acceptance control. For every copied
KNYC fetch it compares API values over complete decoded GRIB intervals. It permits candidate
issues in the preceding 30 hours (covering the slow long-range bundle) and following 30 minutes
(one poll). It does not silently grant a later matching revision to an earlier as-of.
The later investigation explicitly shows whether an unmatched value appears hours later.

Whole-degree Fahrenheit is reconstructed only when the file's packing quantum makes it
unambiguous. Raw Kelvin/Celsius/Fahrenheit values are retained. A 1°F disagreement fails;
floating-point and the declared Kelvin packing precision do not create artificial differences.

The strict control retains **every original API interval**, including clipped starts and
nulls, and requires the same endpoints as well as the same native-F value. Unequal wire
intervals are unresolved, not assumed equivalent. Neither a successful value search nor
a failed interval-label comparison alone establishes different underlying forecasts.

In [4]:
def expand_api_maximum(record):
    layer = record["data"]["properties"]["maxTemperature"]
    assert layer["uom"] == "wmoUnit:degC", f"Unexpected API temperature unit: {layer['uom']}"
    hours = {}
    for interval in layer["values"]:
        start_text, duration_text = interval["validTime"].split("/")
        start = utc(start_text)
        match = re.fullmatch(r"P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?)?", duration_text)
        assert match, f"Unhandled ISO duration: {duration_text}"
        duration = timedelta(
            days=int(match[1] or 0), hours=int(match[2] or 0), minutes=int(match[3] or 0)
        )
        assert duration > timedelta(0) and duration % timedelta(hours=1) == timedelta(0)
        assert start.minute == start.second == 0
        value = interval["value"]
        for offset in range(int(duration.total_seconds() // 3600)):
            hour = start + timedelta(hours=offset)
            assert hour not in hours, "Overlapping API valid periods."
            hours[hour] = value
    return hours


API_HOURS = {r["name"]: expand_api_maximum(r) for r in GRIDS["KNYC"]}


def native_fahrenheit(celsius):
    value = celsius * 1.8 + 32
    assert abs(value - round(value)) < 1e-7, f"API MaxT is not an integer Fahrenheit value: {value}"
    return round(value)


def archive_native_fahrenheit(row):
    value = row["fahrenheit"]
    # NOAA's spreadsheet declares native Fahrenheit, GRIB Kelvin at 0.1 K precision.
    # Require an unambiguous native-degree reconstruction bounded by this file's packing step.
    error_bound = row["packing_quantum_k"] * 0.5 * 1.8 + 5e-5
    assert error_bound < 0.5
    assert abs(value - round(value)) <= error_bound, "Value exceeds GRIB packing precision."
    return round(value)


def interval_value(hours, start, end):
    required = [
        start + timedelta(hours=i) for i in range(int((end - start).total_seconds() // 3600))
    ]
    if not required or any(h not in hours or hours[h] is None for h in required):
        return None
    values = {native_fahrenheit(hours[h]) for h in required}
    return next(iter(values)) if len(values) == 1 else None


def object_matches_fetch(rows, fetch):
    api = API_HOURS[fetch["name"]]
    return all(
        interval_value(api, r["valid_start"], r["valid_end"]) == archive_native_fahrenheit(r)
        for r in rows
    )


# Decode only files whose issue time can overlap the copied window (plus the prior state).
fetch_start = GRIDS["KNYC"][0]["fetched_at"]
fetch_end = GRIDS["KNYC"][-1]["fetched_at"]
OVERLAP_DOWNLOADS = [
    o
    for o in OVERLAP_OBJECTS
    if (fetch_start - timedelta(hours=30) <= o["filename_issue"] <= fetch_end)
]
OVERLAP_ROWS = []
for position, obj in enumerate(OVERLAP_DOWNLOADS):
    OVERLAP_ROWS.extend(decode_object(obj))
    if position % 25 == 0:
        print(f"Decoded overlap object {position + 1}/{len(OVERLAP_DOWNLOADS)}", flush=True)

ARCHIVE_BY_KEY = defaultdict(list)
for row in OVERLAP_ROWS:
    ARCHIVE_BY_KEY[row["key"]].append(row)

# A revision is identifiable by a changed vector of (valid interval, native-F value),
# not by repeated polls or an unchanged NDFD mosaic publication.
LAG_ROWS = []
for key, rows in sorted(ARCHIVE_BY_KEY.items(), key=lambda item: item[1][0]["issue"]):
    issue = rows[0]["issue"]
    if not fetch_start <= issue <= fetch_end:
        continue
    earlier_fetches = [f for f in GRIDS["KNYC"] if f["fetched_at"] < issue]
    unchanged = bool(earlier_fetches) and object_matches_fetch(rows, earlier_fetches[-1])
    later_matches = [
        f for f in GRIDS["KNYC"] if f["fetched_at"] >= issue and object_matches_fetch(rows, f)
    ]
    # The observed window ends at the next differing archive vector for this projection.
    projection = key.split("/")[-1][4:6]
    next_different = [
        rr[0]["issue"]
        for kk, rr in ARCHIVE_BY_KEY.items()
        if (
            kk.split("/")[-1][4:6] == projection
            and rr[0]["issue"] > issue
            and {(r["valid_start"], r["valid_end"], archive_native_fahrenheit(r)) for r in rr}
            != {(r["valid_start"], r["valid_end"], archive_native_fahrenheit(r)) for r in rows}
        )
    ]
    deadline = min(next_different) + timedelta(minutes=30) if next_different else fetch_end
    plausible = [f for f in later_matches if f["fetched_at"] <= deadline]
    first = plausible[0] if plausible else None
    status = (
        "unchanged_value_unidentifiable"
        if unchanged
        else ("observed_changed_vector" if first else "not_seen_before_next_revision_or_window_end")
    )
    LAG_ROWS.append(
        dict(
            key=key,
            issue=issue,
            projection=projection,
            status=status,
            first_seen=first["fetched_at"] if first else None,
            lag_minutes=(first["fetched_at"] - issue).total_seconds() / 60 if first else None,
        )
    )
LAGS = pd.DataFrame(LAG_ROWS)
display(LAGS.groupby(["projection", "status"]).size().rename("issuances"))
display(LAGS)

# Every collected forecast-valid interval is tested. Search the archive around its first
# observed changed value, not arbitrarily far into the future for a coincidental match.
CONTROL = []
seen_api_states = set()
for fetch in GRIDS["KNYC"]:
    api = API_HOURS[fetch["name"]]
    state = tuple(sorted((hour, value) for hour, value in api.items()))
    is_new = state not in seen_api_states
    seen_api_states.add(state)
    for start in sorted(api):
        if start.hour != 12 or api[start] is None:
            continue
        end = start + timedelta(hours=12)
        value = interval_value(api, start, end)
        if value is None:
            continue
        candidates = [
            r
            for r in OVERLAP_ROWS
            if (
                r["valid_start"] == start
                and r["valid_end"] == end
                and fetch["fetched_at"] - timedelta(hours=30)
                <= r["issue"]
                <= fetch["fetched_at"] + timedelta(minutes=30)
            )
        ]
        matches = [r for r in candidates if archive_native_fahrenheit(r) == value]
        nearest = (
            min(candidates, key=lambda r: abs((r["issue"] - fetch["fetched_at"]).total_seconds()))
            if candidates
            else None
        )
        CONTROL.append(
            dict(
                response=fetch["name"],
                fetched_at=fetch["fetched_at"],
                new_api_vector=is_new,
                valid_start=start,
                valid_end=end,
                api_f=value,
                matched=bool(matches),
                archive_f=archive_native_fahrenheit(nearest) if nearest else None,
                key=(
                    min(
                        matches,
                        key=lambda r: abs((r["issue"] - fetch["fetched_at"]).total_seconds()),
                    )["key"]
                    if matches
                    else nearest["key"]
                    if nearest
                    else None
                ),
                candidates=len(candidates),
            )
        )
CONTROL = pd.DataFrame(CONTROL)
MISMATCHES = CONTROL[~CONTROL["matched"]].copy()
display(CONTROL.groupby("matched").size().rename("valid-period comparisons"))
display(MISMATCHES)
CONTROL.to_csv(CACHE / "overlap_control.csv", index=False)
LAGS.to_csv(CACHE / "lag_observations.csv", index=False)
print(
    "Common-interval value search only:",
    "all values found" if MISMATCHES.empty else "missing values found",
)

Decoded overlap object 1/339


Decoded overlap object 26/339


Decoded overlap object 51/339


Decoded overlap object 76/339


Decoded overlap object 101/339


Decoded overlap object 126/339


... (9 outputs left out)

,key,issue,projection,status,first_seen,lag_minutes
0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,2026-11-07 21:47:00+00:00,98,not_seen_before_next_revision_or_window_end,NaT,NaN
1,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072216,2026-11-07 22:16:00+00:00,98,not_seen_before_next_revision_or_window_end,NaT,NaN
2,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072246,2026-11-07 22:46:00+00:00,98,not_seen_before_next_revision_or_window_end,NaT,NaN
3,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072316,2026-11-07 23:16:00+00:00,98,not_seen_before_next_revision_or_window_end,NaT,NaN
4,wmo/maxt/2026/11/07/YGUZ97_KWBN_202611072333,2026-11-07 23:33:00+00:00,97,unchanged_value_unidentifiable,2026-11-07 23:45:45.416421+00:00,12.756940
...,...,...,...,...,...,...
267,wmo/maxt/2026/11/12/YGUZ98_KWBN_202611122316,2026-11-12 23:16:00+00:00,98,not_seen_before_next_revision_or_window_end,NaT,NaN
268,wmo/maxt/2026/11/12/YGUZ97_KWBN_202611122333,2026-11-12 23:33:00+00:00,97,unchanged_value_unidentifiable,2026-11-12 23:43:52.835229+00:00,10.880587
269,wmo/maxt/2026/11/12/YGUZ98_KWBN_202611122346,2026-11-12 23:46:00+00:00,98,unchanged_value_unidentifiable,2026-11-13 00:13:52.713715+00:00,27.878562
270,wmo/maxt/2026/11/13/YGUZ98_KWBN_202611130016,2026-11-13 00:16:00+00:00,98,unchanged_value_unidentifiable,2026-11-13 00:43:52.600792+00:00,27.876680


matched
False     261
True     1665
Name: valid-period comparisons, dtype: int64

,response,fetched_at,new_api_vector,valid_start,valid_end,api_f,matched,archive_f,key,candidates
143,2026-11-08T074545.303607Z.json.gz,2026-11-08 07:45:45.303607+00:00,True,2026-11-11 12:00:00+00:00,2026-11-12 00:00:00+00:00,61,False,63.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6
144,2026-11-08T074545.303607Z.json.gz,2026-11-08 07:45:45.303607+00:00,True,2026-11-12 12:00:00+00:00,2026-11-13 00:00:00+00:00,57,False,58.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6
145,2026-11-08T074545.303607Z.json.gz,2026-11-08 07:45:45.303607+00:00,True,2026-11-13 12:00:00+00:00,2026-11-14 00:00:00+00:00,49,False,51.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6
147,2026-11-08T074545.303607Z.json.gz,2026-11-08 07:45:45.303607+00:00,True,2026-11-15 12:00:00+00:00,2026-11-16 00:00:00+00:00,55,False,NaN,NaN,0
151,2026-11-08T081545.319890Z.json.gz,2026-11-08 08:15:45.319890+00:00,False,2026-11-11 12:00:00+00:00,2026-11-12 00:00:00+00:00,61,False,63.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6
...,...,...,...,...,...,...,...,...,...,...
1855,2026-11-12T201352.644749Z.json.gz,2026-11-12 20:13:52.644749+00:00,False,2026-11-19 12:00:00+00:00,2026-11-20 00:00:00+00:00,62,False,NaN,NaN,0
1859,2026-11-12T204352.583857Z.json.gz,2026-11-12 20:43:52.583857+00:00,False,2026-11-16 12:00:00+00:00,2026-11-17 00:00:00+00:00,52,False,53.0,wmo/maxt/2026/11/12/YGUZ97_KWBN_202611121734,6
1860,2026-11-12T204352.583857Z.json.gz,2026-11-12 20:43:52.583857+00:00,False,2026-11-17 12:00:00+00:00,2026-11-18 00:00:00+00:00,56,False,54.0,wmo/maxt/2026/11/12/YGUZ97_KWBN_202611121734,6
1861,2026-11-12T204352.583857Z.json.gz,2026-11-12 20:43:52.583857+00:00,False,2026-11-18 12:00:00+00:00,2026-11-19 00:00:00+00:00,60,False,59.0,wmo/maxt/2026/11/12/YGUZ97_KWBN_202611121734,5


Common-interval value search only: missing values found


## Availability rule, valid periods, and deliberate failure proofs

Issue time is the **WMO filename timestamp**. Its disagreement with the GRIB reference time
is measured rather than silently substituted. For each issued daily-MaxT field, lag is the
first copied API fetch whose value covers that complete GRIB interval minus issue time.
Signed lags are preserved: an unchanged or repeated value can have been visible days before
a later mosaic issuance. That is not a measured negative publication delay. Unseen fields
and the collection window's censoring are reported explicitly.

Apply the approved rule literally: `L = max(observed lag) + 30 minutes`,
`available_at = filename_issue + L`. Assertions require availability at or after both
`issue + L` and each observed first matching fetch. The empirical rule is not evidence that
all older publication lags were bounded by this short overlap. It never uses S3 Last-Modified
and does not change the production fetch-time contract.

Each required property below has a full-sentence check, deliberate mutation, observed failure,
and restoration. The actual overlap failure is retained and reported separately from the
passing assertion mechanics. Temporary format fixtures repackage a **real** cached envelope
inside the ignored spike cache; no fixtures are added to `tests/fixtures/`.

In [5]:
from zoneinfo import ZoneInfo


def parse_valid_time(valid_time):
    start_text, duration_text = valid_time.split("/")
    match = re.fullmatch(r"P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?)?", duration_text)
    assert match, f"Unsupported duration: {valid_time}"
    start = utc(start_text)
    return start, start + timedelta(
        days=int(match[1] or 0), hours=int(match[2] or 0), minutes=int(match[3] or 0)
    )


def validate_same_period_and_value(archive, api_start, api_end, api_celsius):
    assert archive["valid_start"] == api_start and archive["valid_end"] == api_end, (
        "Archive and API valid periods differ."
    )
    assert archive_native_fahrenheit(archive) == native_fahrenheit(api_celsius), (
        "Archive and API native Fahrenheit maxima differ."
    )


# Preserve every original API interval, including clipped starts and nulls. The common-
# interval diagnostic in the preceding cell does NOT establish identical period semantics.
STRICT_CONTROL = []
for fetch in GRIDS["KNYC"]:
    for entry in fetch["data"]["properties"]["maxTemperature"]["values"]:
        start, end = parse_valid_time(entry["validTime"])
        nearby = [
            r
            for r in OVERLAP_ROWS
            if (
                r["issue"] <= fetch["fetched_at"] + timedelta(minutes=30)
                and r["issue"] >= fetch["fetched_at"] - timedelta(hours=30)
                and r["valid_start"] < end
                and r["valid_end"] > start
            )
        ]
        same_period = [r for r in nearby if r["valid_start"] == start and r["valid_end"] == end]
        same_value = [
            r
            for r in same_period
            if entry["value"] is not None
            and archive_native_fahrenheit(r) == native_fahrenheit(entry["value"])
        ]
        reason = (
            "null_API_value"
            if entry["value"] is None
            else "period_not_reproduced"
            if not same_period
            else "temperature_not_reproduced"
            if not same_value
            else "matched"
        )
        closest = (
            min(nearby, key=lambda r: abs((r["issue"] - fetch["fetched_at"]).total_seconds()))
            if nearby
            else None
        )
        STRICT_CONTROL.append(
            dict(
                response=fetch["name"],
                fetched_at=fetch["fetched_at"],
                api_start=start,
                api_end=end,
                api_hours=(end - start).total_seconds() / 3600,
                api_f=native_fahrenheit(entry["value"]) if entry["value"] is not None else None,
                archive_start=closest["valid_start"] if closest else None,
                archive_end=closest["valid_end"] if closest else None,
                archive_f=archive_native_fahrenheit(closest) if closest else None,
                key=closest["key"] if closest else None,
                result=reason,
            )
        )
STRICT_CONTROL = pd.DataFrame(STRICT_CONTROL)
STRICT_MISMATCHES = STRICT_CONTROL[STRICT_CONTROL.result != "matched"]
STRICT_CONTROL.to_csv(CACHE / "strict_overlap_control.csv", index=False)
display(STRICT_CONTROL.groupby(["result", "api_hours"]).size().rename("API intervals"))
display(STRICT_MISMATCHES)
AWS_CONTROL_PASSED = STRICT_MISMATCHES.empty and MISMATCHES.empty

# Apply the approved definition per issued daily-MaxT field: first collected fetch with
# its value over the entire GRIB interval. Repeated values may have been seen BEFORE issue.
# Preserve those signed lags and distinguish ambiguity from a measured new publication.
literal_lags = []
for archive_row in OVERLAP_ROWS:
    key = archive_row["key"]
    issue = archive_row["issue"]
    if not fetch_start <= issue <= fetch_end:
        continue

    def field_matches(fetch, field=archive_row):
        return interval_value(
            API_HOURS[fetch["name"]], field["valid_start"], field["valid_end"]
        ) == archive_native_fahrenheit(field)

    matches = [f for f in GRIDS["KNYC"] if field_matches(f)]
    first = matches[0] if matches else None
    prior = [f for f in GRIDS["KNYC"] if f["fetched_at"] < issue]
    unchanged = bool(prior) and field_matches(prior[-1])
    literal_lags.append(
        dict(
            key=key,
            issue=issue,
            projection=key.split("/")[-1][4:6],
            valid_start=archive_row["valid_start"],
            valid_end=archive_row["valid_end"],
            native_f=archive_native_fahrenheit(archive_row),
            first_seen=first["fetched_at"] if first else None,
            lag_minutes=(first["fetched_at"] - issue).total_seconds() / 60 if first else None,
            status="not_seen_in_copied_window"
            if first is None
            else "unchanged_or_repeated_value"
            if unchanged
            else "value_seen_before_mosaic_issue"
            if first["fetched_at"] < issue
            else "observed_after_issue",
            left_censored=bool(first and first["name"] == GRIDS["KNYC"][0]["name"]),
        )
    )
LAGS = pd.DataFrame(literal_lags)
LAGS.to_csv(CACHE / "lag_observations.csv", index=False)
OBSERVED_LAGS = LAGS[LAGS.first_seen.notna()].copy()
assert not OBSERVED_LAGS.empty, "No observed lag exists; the rule cannot be calibrated."
L = timedelta(minutes=float(OBSERVED_LAGS.lag_minutes.max())) + timedelta(minutes=30)
LAGS["reconstructed_available_at"] = LAGS.issue + L
LAGS.to_csv(CACHE / "lag_observations.csv", index=False)
display(
    OBSERVED_LAGS.groupby("status").lag_minutes.describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)
display(
    OBSERVED_LAGS.assign(bin_minutes=lambda frame: (frame.lag_minutes // 10 * 10).astype(int))
    .groupby(["status", "bin_minutes"])
    .size()
    .rename("issuances")
)
print("Maximum observed lag:", L - timedelta(minutes=30), "; L:", L)
print("Censored or unidentifiable status counts:", LAGS.status.value_counts().to_dict())
display(
    pd.DataFrame(REFERENCE_DIFFERENCES)
    .drop_duplicates()
    .groupby("difference_minutes")
    .size()
    .rename("GRIB messages")
)


def validate_availability(issue, available_at, delay, first_seen=None):
    assert available_at >= issue + delay, "Reconstructed availability precedes issue + L."
    if first_seen is not None:
        assert available_at >= first_seen, "Reconstructed availability precedes first observation."


for row in OVERLAP_ROWS + PROBE_ROWS:
    row["available_at"] = row["issue"] + L
    validate_availability(row["issue"], row["available_at"], L)
pd.DataFrame(OVERLAP_ROWS).to_csv(CACHE / "overlap_forecasts.csv", index=False)
for row in OBSERVED_LAGS.to_dict("records"):
    validate_availability(row["issue"], row["issue"] + L, L, row["first_seen"])

# Encoded interval labels, not an assumption that a CONUS-wide UTC label overrides the
# documented point-local 07–19 LST MaxT definition. The CLI-day decision stays untouched.
PERIOD_REPORT = []
for station_id in GRIDS:
    timezone = ZoneInfo(POINTS[station_id][0]["data"]["properties"]["timeZone"])
    for archive in [r for r in PROBE_ROWS if r["station"] == station_id]:
        start = archive["valid_start"].astimezone(timezone)
        end = archive["valid_end"].astimezone(timezone)
        PERIOD_REPORT.append(
            dict(
                station=station_id,
                timezone=str(timezone),
                local_start=start.isoformat(),
                local_end=end.isoformat(),
                hours=(end - start).total_seconds() / 3600,
                documented_MaxT_definition="07:00–19:00 local standard time (NWSI 10-201)",
                relation_to_midnight_day=(
                    "Encoded GRIB label shown; documented MaxT is daytime-only, "
                    "not a full CLI climate day"
                ),
            )
        )
display(pd.DataFrame(PERIOD_REPORT))

PROOFS = []


def expect_failure(property_name, mutation, action):
    try:
        action()
    except (AssertionError, ValueError, zstandard.ZstdError) as error:
        PROOFS.append(
            dict(property=property_name, mutation=mutation, failure=str(error), restored=True)
        )
        print(f"EXPECTED FAILURE: {property_name}: {error}")
    else:
        raise AssertionError(f"Mutation did not fire: {property_name}: {mutation}")


# A same-period pair uses the original API value restricted to the GRIB common interval.
# This proves the assertion, not equivalence of the unequal original API/GRIB endpoints.
good_archive = next(
    r
    for r in OVERLAP_ROWS
    if any(
        interval_value(API_HOURS[f["name"]], r["valid_start"], r["valid_end"])
        == archive_native_fahrenheit(r)
        for f in GRIDS["KNYC"]
    )
)
good_c = (archive_native_fahrenheit(good_archive) - 32) / 1.8
validate_same_period_and_value(
    good_archive, good_archive["valid_start"], good_archive["valid_end"], good_c
)
bad_archive = {**good_archive, "fahrenheit": good_archive["fahrenheit"] + 1}
expect_failure(
    "Every collected issuance has the same archived maximum for the same valid period",
    "Raise one archived maximum by 1 F",
    lambda: validate_same_period_and_value(
        bad_archive, good_archive["valid_start"], good_archive["valid_end"], good_c
    ),
)
expect_failure(
    "Every collected issuance has the same archived maximum for the same valid period",
    "Extend one API period by one hour",
    lambda: validate_same_period_and_value(
        good_archive,
        good_archive["valid_start"],
        good_archive["valid_end"] + timedelta(hours=1),
        good_c,
    ),
)
sample_lag = OBSERVED_LAGS.iloc[0]
expect_failure(
    "Every reconstructed availability is no earlier than issue plus L and first observation",
    "Backdate availability by one second",
    lambda: validate_availability(
        sample_lag.issue, sample_lag.issue + L - timedelta(seconds=1), L, sample_lag.first_seen
    ),
)
expect_failure(
    "Every reconstructed availability is no earlier than issue plus L and first observation",
    "Move first observation after reconstructed availability",
    lambda: validate_availability(
        sample_lag.issue, sample_lag.issue + L, L, sample_lag.issue + L + timedelta(seconds=1)
    ),
)
location = next(iter(GRID_LOCATIONS.values()))
expect_failure(
    "Every sampled NDFD cell contains the collected API polygon centroid",
    "Shift the selected column by one",
    lambda: validate_cell(
        location["row"],
        location["column"] + 1,
        location["fractional_y"],
        location["fractional_x"],
        location["ny"],
        location["nx"],
    ),
)


def validate_inventory(objects, expected_days):
    keys = [obj["key"] for obj in objects]
    assert len(set(keys)) == len(keys), "An issuance is duplicated in the inventory."
    actual_days = {o["filename_issue"].date() for o in objects}
    assert actual_days == set(expected_days), (
        f"Missing inventory days: {sorted(set(expected_days) - actual_days)}"
    )


validate_inventory(OVERLAP_OBJECTS, OVERLAP_INVENTORY_DAYS)
expect_failure(
    "Every sampled coverage day and issuance is explicitly accounted for",
    "Delete a sampled inventory day",
    lambda: validate_inventory(
        [o for o in OVERLAP_OBJECTS if o["filename_issue"].date() != OVERLAP_DAYS[1]],
        OVERLAP_INVENTORY_DAYS,
    ),
)
expect_failure(
    "Every sampled coverage day and issuance is explicitly accounted for",
    "Duplicate one issuance",
    lambda: validate_inventory(OVERLAP_OBJECTS + [OVERLAP_OBJECTS[0]], OVERLAP_INVENTORY_DAYS),
)


def validate_period(start, end):
    assert end - start == timedelta(hours=12), "MaxT statistical period is not twelve hours."
    assert start.hour == 12 and end.hour == 0, "Unexpected CONUS MaxT UTC boundaries."


for row in OVERLAP_ROWS:
    validate_period(row["valid_start"], row["valid_end"])
expect_failure(
    "Every maximum temperature retains its decoded statistical valid period",
    "Replace a twelve-hour period with twenty-four hours",
    lambda: validate_period(
        good_archive["valid_start"], good_archive["valid_start"] + timedelta(hours=24)
    ),
)
expect_failure(
    "Every maximum temperature retains its decoded statistical valid period",
    "Shift both boundaries by one hour",
    lambda: validate_period(
        good_archive["valid_start"] + timedelta(hours=1),
        good_archive["valid_end"] + timedelta(hours=1),
    ),
)

# Repackage an unchanged real envelope in both production archive forms.
FORMAT_FIXTURES = CACHE / "format_checks"
real_source = next(COLLECTED.rglob("*.json.gz"))
real_payload = gzip.decompress(real_source.read_bytes())
fixture_day = FORMAT_FIXTURES / real_source.parent.name
fixture_day.mkdir(parents=True, exist_ok=True)
(fixture_day / real_source.name).write_bytes(real_source.read_bytes())


def write_fixture_archive(destination, payload):
    with (
        destination.open("wb") as target,
        zstandard.ZstdCompressor(write_checksum=True).stream_writer(target) as compressed,
        tarfile.open(fileobj=compressed, mode="w|") as container,
    ):
        member = tarfile.TarInfo(real_source.name.removesuffix(".gz"))
        member.size = len(payload)
        container.addfile(member, io.BytesIO(payload))


archive_hour = real_source.name[:13]
regular_fixture = fixture_day / (archive_hour + ".tar.zst")
late_fixture = fixture_day / (archive_hour + ".late-1.tar.zst")
write_fixture_archive(regular_fixture, real_payload)
write_fixture_archive(late_fixture, real_payload)
fixture_records, fixture_counts = read_inputs(FORMAT_FIXTURES, CACHE / "format_checks.sqlite")
assert len(fixture_records) == 1 and fixture_counts["duplicates"].sum() == 2
assert fixture_counts[["loose", "compacted", "late"]].iloc[0].tolist() == [1, 1, 1]
conflicting_envelope = json.loads(real_payload)
conflicting_envelope["body"] += " "
write_fixture_archive(late_fixture, json.dumps(conflicting_envelope).encode())
expect_failure(
    "Repeated original filenames have identical bodies in every input format",
    "Change the same-name late archive body",
    lambda: read_inputs(FORMAT_FIXTURES, CACHE / "format_conflict.sqlite"),
)
write_fixture_archive(late_fixture, real_payload)
restored_records, _ = read_inputs(FORMAT_FIXTURES, CACHE / "format_restored.sqlite")
assert len(restored_records) == 1
display(pd.DataFrame(PROOFS))
print(
    "Property 1 real-data status:",
    "PASS" if AWS_CONTROL_PASSED else "FAIL — exact-period equivalence is unproved",
)
print(
    "Property 2 status: PASS for the approved empirical rule; "
    "future historical lag is an assumption."
)

result                 api_hours
period_not_reproduced  1.0            36
                       8.0            30
                       9.0             6
                       10.0           22
                       12.0           16
                       13.0         1926
Name: API intervals, dtype: int64

,response,fetched_at,api_start,api_end,api_hours,api_f,archive_start,archive_end,archive_f,key,result
0,2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-07 15:00:00+00:00,2026-11-08 01:00:00+00:00,10.0,63,2026-11-07 12:00:00+00:00,2026-11-08 00:00:00+00:00,63.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,period_not_reproduced
1,2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-08 12:00:00+00:00,2026-11-09 01:00:00+00:00,13.0,62,2026-11-08 12:00:00+00:00,2026-11-09 00:00:00+00:00,62.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,period_not_reproduced
2,2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-09 12:00:00+00:00,2026-11-10 01:00:00+00:00,13.0,58,2026-11-09 12:00:00+00:00,2026-11-10 00:00:00+00:00,58.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,period_not_reproduced
3,2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-10 12:00:00+00:00,2026-11-11 01:00:00+00:00,13.0,61,2026-11-10 12:00:00+00:00,2026-11-11 00:00:00+00:00,61.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,period_not_reproduced
4,2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-11 12:00:00+00:00,2026-11-12 01:00:00+00:00,13.0,63,2026-11-11 12:00:00+00:00,2026-11-12 00:00:00+00:00,63.0,wmo/maxt/2026/11/07/YGUZ97_KWBN_202611072133,period_not_reproduced
...,...,...,...,...,...,...,...,...,...,...,...
2031,2026-11-13T011352.643025Z.json.gz,2026-11-13 01:13:52.643025+00:00,2026-11-15 12:00:00+00:00,2026-11-16 01:00:00+00:00,13.0,49,2026-11-15 12:00:00+00:00,2026-11-16 00:00:00+00:00,49.0,wmo/maxt/2026/11/13/YGUZ98_KWBN_202611130046,period_not_reproduced
2032,2026-11-13T011352.643025Z.json.gz,2026-11-13 01:13:52.643025+00:00,2026-11-16 12:00:00+00:00,2026-11-17 01:00:00+00:00,13.0,52,2026-11-16 12:00:00+00:00,2026-11-17 00:00:00+00:00,52.0,wmo/maxt/2026/11/12/YGUZ97_KWBN_202611122333,period_not_reproduced
2033,2026-11-13T011352.643025Z.json.gz,2026-11-13 01:13:52.643025+00:00,2026-11-17 12:00:00+00:00,2026-11-18 01:00:00+00:00,13.0,56,2026-11-17 12:00:00+00:00,2026-11-18 00:00:00+00:00,56.0,wmo/maxt/2026/11/12/YGUZ97_KWBN_202611122333,period_not_reproduced
2034,2026-11-13T011352.643025Z.json.gz,2026-11-13 01:13:52.643025+00:00,2026-11-18 12:00:00+00:00,2026-11-19 01:00:00+00:00,13.0,60,2026-11-18 12:00:00+00:00,2026-11-19 00:00:00+00:00,60.0,wmo/maxt/2026/11/12/YGUZ97_KWBN_202611122333,period_not_reproduced


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
status,,,,,,,,,,,
observed_after_issue,8.0,29.035929,1.332763,26.876606,29.035447,29.755120,29.755979,29.756568,29.756568,29.756568,29.756568
unchanged_or_repeated_value,828.0,-2111.844067,1769.376721,-6260.144909,-3630.144909,-1650.144909,-480.244910,-180.144909,-90.902399,-1.409100,-0.243432
value_seen_before_mosaic_issue,16.0,-3156.000793,2059.081059,-5850.144909,-4357.644909,-4260.694916,-1020.169540,-165.243432,-142.743432,-124.743432,-120.243432


status                          bin_minutes
observed_after_issue             20            8
unchanged_or_repeated_value     -6270          1
                                -6250          1
                                -6210          1
                                -6190          1
                                              ..
value_seen_before_mosaic_issue  -1300          1
                                -220           1
                                -190           1
                                -160           1
                                -130           1
Name: issuances, Length: 299, dtype: int64

Maximum observed lag: 0:29:45.394069 ; L: 0:59:45.394069
Censored or unidentifiable status counts: {'unchanged_or_repeated_value': 828, 'value_seen_before_mosaic_issue': 16, 'not_seen_in_copied_window': 14, 'observed_after_issue': 8}


difference_minutes
13.0      9
14.0    248
15.0     26
24.0     22
25.0      2
26.0      8
27.0     25
Name: GRIB messages, dtype: int64

,station,timezone,local_start,local_end,hours,documented_MaxT_definition,relation_to_midnight_day
0,KAUS,America/Chicago,2026-11-06T06:00:00-06:00,2026-11-06T18:00:00-06:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
1,KAUS,America/Chicago,2026-11-07T06:00:00-06:00,2026-11-07T18:00:00-06:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
2,KAUS,America/Chicago,2026-11-08T06:00:00-06:00,2026-11-08T18:00:00-06:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
3,KMDW,America/Chicago,2026-11-06T06:00:00-06:00,2026-11-06T18:00:00-06:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
4,KMDW,America/Chicago,2026-11-07T06:00:00-06:00,2026-11-07T18:00:00-06:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
5,KMDW,America/Chicago,2026-11-08T06:00:00-06:00,2026-11-08T18:00:00-06:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
6,KDEN,America/Denver,2026-11-06T05:00:00-07:00,2026-11-06T17:00:00-07:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
7,KDEN,America/Denver,2026-11-07T05:00:00-07:00,2026-11-07T17:00:00-07:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
8,KDEN,America/Denver,2026-11-08T05:00:00-07:00,2026-11-08T17:00:00-07:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...
9,KLAX,America/Los_Angeles,2026-11-06T04:00:00-08:00,2026-11-06T16:00:00-08:00,12.0,07:00–19:00 local standard time (NWSI 10-201),Encoded GRIB label shown; documented MaxT is d...


EXPECTED FAILURE: Every collected issuance has the same archived maximum for the same valid period: Archive and API native Fahrenheit maxima differ.
EXPECTED FAILURE: Every collected issuance has the same archived maximum for the same valid period: Archive and API valid periods differ.
EXPECTED FAILURE: Every reconstructed availability is no earlier than issue plus L and first observation: Reconstructed availability precedes issue + L.
EXPECTED FAILURE: Every reconstructed availability is no earlier than issue plus L and first observation: Reconstructed availability precedes first observation.
EXPECTED FAILURE: Every sampled NDFD cell contains the collected API polygon centroid: Selected NDFD cell does not contain the collected API polygon centroid.
EXPECTED FAILURE: Every sampled coverage day and issuance is explicitly accounted for: Missing inventory days: [datetime.date(2026, 11, 8)]
EXPECTED FAILURE: Every sampled coverage day and issuance is explicitly accounted for: An issuance i

,property,mutation,failure,restored
0,Every collected issuance has the same archived...,Raise one archived maximum by 1 F,Archive and API native Fahrenheit maxima differ.,True
1,Every collected issuance has the same archived...,Extend one API period by one hour,Archive and API valid periods differ.,True
2,Every reconstructed availability is no earlier...,Backdate availability by one second,Reconstructed availability precedes issue + L.,True
3,Every reconstructed availability is no earlier...,Move first observation after reconstructed ava...,Reconstructed availability precedes first obse...,True
4,Every sampled NDFD cell contains the collected...,Shift the selected column by one,Selected NDFD cell does not contain the collec...,True
5,Every sampled coverage day and issuance is exp...,Delete a sampled inventory day,"Missing inventory days: [datetime.date(2026, 1...",True
6,Every sampled coverage day and issuance is exp...,Duplicate one issuance,An issuance is duplicated in the inventory.,True
7,Every maximum temperature retains its decoded ...,Replace a twelve-hour period with twenty-four ...,MaxT statistical period is not twelve hours.,True
8,Every maximum temperature retains its decoded ...,Shift both boundaries by one hour,Unexpected CONUS MaxT UTC boundaries.,True
9,Repeated original filenames have identical bod...,Change the same-name late archive body,Different bodies share original name 2026-11-1...,True


Property 1 real-data status: FAIL — exact-period equivalence is unproved
Property 2 status: PASS for the approved empirical rule; future historical lag is an assumption.


## Thirty-day AWS survey and two full-backfill estimates

These historical values are **unvalidated as reproductions of API forecasts** whenever the
overlap fails. This survey verifies accessible files, decoding, sampled point values and
cadence; it authorizes no training or full backfill.

The fixed 30-day list spans the documented AWS range and includes leap day. Two representative
GRIB files per sampled day exercise both forecast-range bundles. Metadata inventories include
lookback days so selection can use the latest `issue + L <= decision` at **00, 06, 12, 18 UTC**.
Keep one latest file **per forecast-range bundle** at each decision, deduplicated across
decisions and all seven stations: up to eight shared CONUS objects/day. Compare with every
operational issuance. No every-issuance historical payload backfill is executed.

Costs extrapolate the stratified sample's object counts and compressed bytes using measured
Mac transfer/decode times and the one-second request policy. They are planning estimates,
not confidence bounds; retries, future product changes and unsampled outages can change them.
The final partial day and unsampled point coverage remain explicit.

In [6]:
# This is a format/coverage survey, not permission to train on historical forecasts.
# When strict overlap fails, all historical values remain explicitly UNVALIDATED.
AWS_START = date(2020, 4, 16)
AWS_END = max(OVERLAP_DAYS)
HISTORICAL_DAYS = [
    date.fromordinal(round(number))
    for number in np.linspace(AWS_START.toordinal(), (AWS_END - timedelta(days=1)).toordinal(), 30)
]
HISTORICAL_DAYS[18] = date(2024, 2, 29)
HISTORICAL_DAYS = sorted(set(HISTORICAL_DAYS))
assert len(HISTORICAL_DAYS) == 30 and min(HISTORICAL_DAYS) >= AWS_START
print("Frozen historical sample:", [d.isoformat() for d in HISTORICAL_DAYS])
print(
    "Historical NWS equivalence:",
    "VALIDATED" if AWS_CONTROL_PASSED else "UNVALIDATED; AWS inventory/format survey only",
)

INVENTORIES = {
    day: [obj for obj in OVERLAP_OBJECTS if obj["filename_issue"].date() == day]
    for day in OVERLAP_INVENTORY_DAYS
}
HISTORY_ROWS = []
HISTORY_COUNTS = []
for day in HISTORICAL_DAYS:
    if day not in INVENTORIES:
        INVENTORIES[day] = day_inventory(day)
    objects = INVENTORIES[day]
    HISTORY_COUNTS.append(
        dict(
            day=day.isoformat(),
            short_range=sum(o["projection"] == "98" for o in objects),
            long_range=sum(o["projection"] == "97" for o in objects),
            bytes=sum(o["size"] for o in objects),
            first_issue=min((o["filename_issue"] for o in objects), default=None),
            last_issue=max((o["filename_issue"] for o in objects), default=None),
        )
    )
    # Two representative files per day prove point extraction and both forecast ranges.
    for projection in ("98", "97"):
        family = [o for o in objects if o["projection"] == projection]
        if family:
            representative = min(family, key=lambda o: o["filename_issue"])
            decoded_history = decode_object(representative)
            for forecast in decoded_history:
                forecast["available_at"] = forecast["issue"] + L
                validate_availability(forecast["issue"], forecast["available_at"], L)
            HISTORY_ROWS.extend(decoded_history)
    print("Surveyed historical day", day, flush=True)
HISTORY_COUNTS = pd.DataFrame(HISTORY_COUNTS)
display(HISTORY_COUNTS)
display(pd.DataFrame(HISTORY_ROWS))
pd.DataFrame(HISTORY_ROWS).to_csv(CACHE / "historical_forecasts_sample.csv", index=False)
print(
    "Sample days missing short-range files:",
    HISTORY_COUNTS.loc[HISTORY_COUNTS.short_range == 0, "day"].tolist(),
)
print(
    "Sample days missing long-range files:",
    HISTORY_COUNTS.loc[HISTORY_COUNTS.long_range == 0, "day"].tolist(),
)

# List observed object-cadence gaps; the docs promise no fixed issuance schedule.
CADENCE_GAPS = []
for day, objects in sorted(INVENTORIES.items()):
    for projection, cadence in (("98", timedelta(minutes=30)), ("97", None)):
        times = sorted(o["filename_issue"] for o in objects if o["projection"] == projection)
        for earlier, later in zip(times, times[1:], strict=False):
            if cadence and later - earlier > cadence + timedelta(minutes=5):
                CADENCE_GAPS.append(
                    dict(
                        day=day.isoformat(),
                        projection=projection,
                        start=earlier,
                        end=later,
                        minutes=(later - earlier).total_seconds() / 60,
                    )
                )
display(pd.DataFrame(CADENCE_GAPS))

# Per-station inventory coverage refers to the shared CONUS product, with point validation
# restricted to KNYC historical samples and the seven-station overlap probe above.
coverage_rows = []
for station_id in GRIDS:
    coverage_rows.append(
        dict(
            station=station_id,
            product="Operational 2.5 km CONUS MaxT (YGUZ97/98)",
            first_sampled_day=min(HISTORICAL_DAYS).isoformat(),
            last_sampled_day=max(OVERLAP_DAYS).isoformat(),
            sampled_inventory_days=len(INVENTORIES),
            historical_point_sample_days=len(HISTORICAL_DAYS) if station_id == "KNYC" else 0,
            missing_sample_days=[d.isoformat() for d, objs in INVENTORIES.items() if not objs],
            unsampled_days="Not inventoried; no claim of gap-free continuous coverage",
            equivalence="PASS" if AWS_CONTROL_PASSED and station_id == "KNYC" else "UNVALIDATED",
        )
    )
COVERAGE = pd.DataFrame(coverage_rows)
display(COVERAGE)


def validate_coverage_report(inventories, requested_days, reported_gaps):
    assert set(requested_days) <= set(inventories), "A requested inventory day disappeared."
    actual_gaps = {day.isoformat() for day in requested_days if not inventories[day]}
    assert actual_gaps == set(reported_gaps), "A coverage gap was not explicitly reported."


requested_days = sorted(set(HISTORICAL_DAYS + OVERLAP_INVENTORY_DAYS))
reported_gaps = [day.isoformat() for day in requested_days if not INVENTORIES[day]]
validate_coverage_report(INVENTORIES, requested_days, reported_gaps)
hidden_gap_inventories = dict(INVENTORIES)
hidden_gap_inventories[HISTORICAL_DAYS[0]] = []
expect_failure(
    "Every sampled coverage day and issuance is explicitly accounted for",
    "Hide a real-format empty-day inventory from the gap list",
    lambda: validate_coverage_report(hidden_gap_inventories, requested_days, reported_gaps),
)
validate_coverage_report(INVENTORIES, requested_days, reported_gaps)

# Four fixed UTC decisions share downloaded CONUS objects across all seven stations.
DECISION_HOURS = (0, 6, 12, 18)
print("Decision times: 00:00, 06:00, 12:00, 18:00 UTC (six-hourly throughout the day).")
print("For KNYC these are 19:00 prior day, 01:00, 07:00, 13:00 EST; one hour later in EDT.")


def selected_at_decisions(day, objects, delay):
    selected = {}
    missing = []
    for hour in DECISION_HOURS:
        decision = datetime.combine(day, datetime.min.time(), tzinfo=UTC) + timedelta(hours=hour)
        for projection in ("98", "97"):
            eligible = [
                o
                for o in objects
                if (o["projection"] == projection and o["filename_issue"] + delay <= decision)
            ]
            if not eligible:
                missing.append(f"{decision.isoformat()} projection={projection}")
            else:
                latest = max(eligible, key=lambda o: o["filename_issue"])
                assert latest["filename_issue"] + delay <= decision
                selected[latest["key"]] = latest
    return list(selected.values()), missing


# Additional lookback listings support selection; they are not additional historical GRIB
# downloads. No complete historical inventory or full backfill is performed.
lookback_days = math.ceil(L.total_seconds() / 86400) + 1
COST_DAYS = [d for d in HISTORICAL_DAYS if d >= AWS_START + timedelta(days=lookback_days)]
COST_ROWS = []
for day in COST_DAYS:
    surrounding = []
    for offset in range(lookback_days + 1):
        previous_day = day - timedelta(days=offset)
        if previous_day not in INVENTORIES:
            INVENTORIES[previous_day] = day_inventory(previous_day)
        surrounding.extend(INVENTORIES[previous_day])
    selected, missing = selected_at_decisions(day, surrounding, L)
    today = INVENTORIES[day]
    COST_ROWS.append(
        dict(
            day=day.isoformat(),
            selected_objects=len(selected),
            selected_bytes=sum(o["size"] for o in selected),
            every_objects=len(today),
            every_bytes=sum(o["size"] for o in today),
            missing_decisions=missing,
        )
    )
DAILY_COST = pd.DataFrame(COST_ROWS)
display(DAILY_COST)
assert not DAILY_COST.empty
print("Incomplete decision selections:", [m for ms in DAILY_COST.missing_decisions for m in ms])

full_days = (AWS_END - AWS_START).days + 1
get_observations = {r["url"]: r for r in REQUEST_LOG if "/YGUZ" in r["url"] and r["status"] == 200}
seconds_per_byte = sum(r["seconds"] for r in get_observations.values()) / sum(
    r["size"] for r in get_observations.values()
)
decode_seconds_per_byte = sum(r["seconds"] for r in DECODE_LOG) / sum(
    r["bytes"] for r in DECODE_LOG
)
ESTIMATES = []
for label, prefix in (
    ("Latest available at four UTC decisions", "selected"),
    ("Every archived issuance", "every"),
):
    mean_objects = float(DAILY_COST[f"{prefix}_objects"].mean())
    mean_bytes = float(DAILY_COST[f"{prefix}_bytes"].mean())
    objects = math.ceil(full_days * mean_objects)
    total_bytes = full_days * mean_bytes
    # One or more listings/day, then one GET per distinct selected object. Full-day sizes
    # are measured; retry traffic and future product changes require additional allowance.
    listing_requests = full_days  # Observed per-day selected-family listing fits one page.
    download_seconds = max(objects, total_bytes * seconds_per_byte)
    decode_seconds = total_bytes * decode_seconds_per_byte
    ESTIMATES.append(
        dict(
            strategy=label,
            stations=7,
            days=full_days,
            mean_objects_per_day=mean_objects,
            mean_MiB_per_day=mean_bytes / 1024**2,
            estimated_GiB=total_bytes / 1024**3,
            GET_requests=objects,
            listing_requests=listing_requests,
            total_requests=objects + listing_requests,
            download_hours=download_seconds / 3600,
            decode_hours=decode_seconds / 3600,
            estimated_serial_hours=(download_seconds + decode_seconds + listing_requests) / 3600,
            caveat=(
                "Sample extrapolation; shared CONUS downloads serve seven stations; "
                "equivalence unvalidated"
            )
            if not AWS_CONTROL_PASSED
            else "Sample extrapolation; shared CONUS objects",
        )
    )
ESTIMATES = pd.DataFrame(ESTIMATES)
display(ESTIMATES)
display(pd.DataFrame(DECODE_LOG).describe())
print("Measured maximum process RSS MiB:", max(r["peak_rss_mib"] for r in DECODE_LOG))
HISTORY_COUNTS.to_csv(CACHE / "historical_inventory_sample.csv", index=False)
DAILY_COST.to_csv(CACHE / "four_decision_daily_cost.csv", index=False)
ESTIMATES.to_csv(CACHE / "backfill_estimates.csv", index=False)

Frozen historical sample: ['2020-04-16', '2020-07-08', '2020-09-29', '2020-12-20', '2021-03-13', '2021-06-04', '2021-08-26', '2021-11-17', '2022-02-07', '2022-05-01', '2022-07-23', '2022-10-14', '2023-01-05', '2023-03-28', '2023-06-19', '2023-09-10', '2023-12-02', '2024-02-22', '2024-02-29', '2024-08-06', '2024-10-28', '2025-01-19', '2025-04-11', '2025-07-03', '2025-09-24', '2025-12-16', '2026-03-09', '2026-05-30', '2026-08-21', '2026-11-12']
Historical NWS equivalence: UNVALIDATED; AWS inventory/format survey only
Surveyed historical day 2020-04-16


Surveyed historical day 2020-07-08


Surveyed historical day 2020-09-29


Surveyed historical day 2020-12-20


Surveyed historical day 2021-03-13


Surveyed historical day 2021-06-04


... (31 outputs left out)

Incomplete decision selections: []


,strategy,stations,days,mean_objects_per_day,mean_MiB_per_day,estimated_GiB,GET_requests,listing_requests,total_requests,download_hours,decode_hours,estimated_serial_hours,caveat
0,Latest available at four UTC decisions,7,2403,7.965517,28.521923,66.931817,19142,2403,21545,7.676987,0.226036,8.570523,Sample extrapolation; shared CONUS downloads s...
1,Every archived issuance,7,2403,52.758621,165.498589,388.372178,126779,2403,129182,44.545752,1.311574,46.524826,Sample extrapolation; shared CONUS downloads s...


,seconds,bytes,peak_rss_mib
count,400.000000,4.000000e+02,400.000000
mean,0.037993,3.355451e+06,401.411133
std,0.005343,4.333920e+05,32.179759
min,0.033738,2.991516e+06,206.781250
25%,0.034611,3.114904e+06,409.140625
50%,0.035006,3.127295e+06,410.906250
75%,0.040526,3.152441e+06,410.906250
max,0.055506,4.277884e+06,410.906250


Measured maximum process RSS MiB: 410.90625


## Complete findings, mismatch tables, coverage limits, and decision drafts

The directory-only census covers every month in the AWS range. It identifies missing MaxT
directories across all domains; it does **not** certify an operational CONUS field or
non-null station value on every unsampled day. Per-station edge probes and the KNYC sample
are shown separately. This keeps coverage claims within the requested limited backfill.

All strict mismatches, common-interval diagnostic mismatches, lag observations (including
unobserved cases), format accounting and mutation proofs are embedded below and also saved
as CSV/JSON in the ignored cache. No failed real-data invariant is weakened to obtain a pass.
No NCEI or MOS archive is accessed. The two draft decisions are **not written**.

In [7]:
from IPython.display import HTML, Markdown

# Directory-only census: inexpensive metadata, no full historical GRIB backfill.
ARCHIVE_DIRECTORY_DAYS = set()
for month in pd.period_range(AWS_START, AWS_END, freq="M"):
    _, prefixes = list_objects(f"wmo/maxt/{month.year:04d}/{month.month:02d}/", delimiter="/")
    for prefix in prefixes:
        match = re.fullmatch(r"wmo/maxt/(\d{4})/(\d{2})/(\d{2})/", prefix)
        assert match, f"Unexpected daily directory: {prefix}"
        ARCHIVE_DIRECTORY_DAYS.add(date(int(match[1]), int(match[2]), int(match[3])))
expected_directory_days = {AWS_START + timedelta(days=i) for i in range(full_days)}
MISSING_DIRECTORY_DAYS = sorted(expected_directory_days - ARCHIVE_DIRECTORY_DAYS)
print("Daily MaxT directory census (all domains; not proof of CONUS point values):")
print("First:", min(ARCHIVE_DIRECTORY_DAYS), "last:", max(ARCHIVE_DIRECTORY_DAYS))
print("Missing directories:", [d.isoformat() for d in MISSING_DIRECTORY_DAYS])
print(
    "Unsampled CONUS daily inventories:",
    full_days - len(set(INVENTORIES) & expected_directory_days),
)

# Cheap point checks at each edge use existing cached CONUS objects once for seven points.
EDGE_ROWS = []
for edge_obj in (
    min(INVENTORIES[AWS_START], key=lambda o: o["filename_issue"]),
    max(OVERLAP_OBJECTS, key=lambda o: o["filename_issue"]),
):
    EDGE_ROWS.extend(decode_object(edge_obj, tuple(GRIDS)))
EDGE_ROWS = pd.DataFrame(EDGE_ROWS)
EDGE_ROWS["available_at"] = EDGE_ROWS.issue + L
display(
    EDGE_ROWS.groupby("station").agg(
        first_tested_issue=("issue", "min"),
        last_tested_issue=("issue", "max"),
        tested_fields=("key", "size"),
    )
)

# Investigate the failed time-window search without silently granting later revisions
# to an earlier as-of. Later matching values are diagnostic, never used as past inputs.
investigation = []
for record in MISMATCHES.to_dict("records"):
    fields = [
        r
        for r in OVERLAP_ROWS
        if (r["valid_start"] == record["valid_start"] and r["valid_end"] == record["valid_end"])
    ]
    matching_fields = [r for r in fields if archive_native_fahrenheit(r) == record["api_f"]]
    nearest = (
        min(matching_fields, key=lambda r: abs((r["issue"] - record["fetched_at"]).total_seconds()))
        if matching_fields
        else None
    )
    investigation.append(
        {
            **record,
            "matching_value_elsewhere_in_overlap": bool(matching_fields),
            "closest_matching_issue": nearest["issue"] if nearest else None,
            "closest_matching_key": nearest["key"] if nearest else None,
            "matching_issue_minus_fetch_minutes": (
                nearest["issue"] - record["fetched_at"]
            ).total_seconds()
            / 60
            if nearest
            else None,
            "all_values_for_period_in_overlap": sorted(
                {archive_native_fahrenheit(r) for r in fields}
            ),
        }
    )
INVESTIGATION = pd.DataFrame(investigation)
INVESTIGATION.to_csv(CACHE / "value_mismatch_investigation.csv", index=False)
display(
    INVESTIGATION.groupby("matching_value_elsewhere_in_overlap").size().rename("fetch/period pairs")
)
display(
    INVESTIGATION.drop_duplicates(["valid_start", "api_f"])[
        [
            "fetched_at",
            "valid_start",
            "api_f",
            "closest_matching_issue",
            "matching_issue_minus_fetch_minutes",
            "all_values_for_period_in_overlap",
        ]
    ]
)

# Show every mismatch and every lag, without pandas' default row truncation.
display(HTML("<h4>Every strict-period mismatch</h4>" + STRICT_MISMATCHES.to_html(index=False)))
display(
    HTML(
        "<h4>Every common-interval value-search mismatch (diagnostic)</h4>"
        + INVESTIGATION.to_html(index=False)
    )
)
display(HTML("<h4>Every observed or censored lag</h4>" + LAGS.to_html(index=False)))
display(
    HTML(
        "<h4>Every mutation failure and restoration</h4>"
        + pd.DataFrame(PROOFS).to_html(index=False)
    )
)

reference_checks = pd.DataFrame(OVERLAP_ROWS + HISTORY_ROWS).drop_duplicates("key")
reference_checks["difference_minutes"] = [
    (r.grib_reference - r.issue).total_seconds() / 60 for r in reference_checks.itertuples()
]
display(reference_checks.groupby("difference_minutes").size().rename("files"))
print(
    "Filename/reference disagreements:",
    sum(reference_checks.difference_minutes != 0),
    "/",
    len(reference_checks),
)
print(
    "Fields that reference+leadTime would mislabel:",
    sum(r["lead_time_would_mislabel"] for r in OVERLAP_ROWS),
)
directory_anomalies = [
    o for objects in INVENTORIES.values() for o in objects if not o["issue_date_matches_directory"]
]
display(pd.DataFrame(directory_anomalies))

poll_gaps = []
poll_counts = []
for station, fetches in GRIDS.items():
    for previous, current in zip(fetches, fetches[1:], strict=False):
        gap_minutes = (current["fetched_at"] - previous["fetched_at"]).total_seconds() / 60
        if gap_minutes > 35:  # Thirty-minute timer plus five-minute diagnostic tolerance.
            poll_gaps.append(
                dict(
                    station=station,
                    start=previous["fetched_at"],
                    end=current["fetched_at"],
                    minutes=gap_minutes,
                )
            )
    for day in OVERLAP_DAYS:
        daily = [f for f in fetches if f["fetched_at"].date() == day]
        poll_counts.append(
            dict(
                station=station,
                day=day,
                grid_fetches=len(daily),
                unique_max_vectors=len(
                    {
                        json.dumps(f["data"]["properties"]["maxTemperature"], sort_keys=True)
                        for f in daily
                    }
                ),
            )
        )
display(pd.DataFrame(poll_counts))
display(pd.DataFrame(poll_gaps))

# Historical policy checks apply to every survey field and selected decision.
for row in HISTORY_ROWS:
    validate_availability(row["issue"], row["available_at"], L)
assert set(EDGE_ROWS.station) == {s["station_id"] for s in STATIONS}
assert len(PROOFS) >= 11 and all(p["restored"] for p in PROOFS)


def assert_real_overlap_control():
    assert AWS_CONTROL_PASSED, (
        f"AWS overlap failed: {len(STRICT_MISMATCHES)}/{len(STRICT_CONTROL)} exact-period "
        f"comparisons unproved; {len(MISMATCHES)}/{len(CONTROL)} common-interval "
        "value searches unmatched. No NCEI/MOS fallback was accessed."
    )


try:
    assert_real_overlap_control()
except AssertionError as real_failure:
    print("REAL DATA FAILURE (retained, not weakened):", real_failure)

max_lag = float(OBSERVED_LAGS.lag_minutes.max())
positive_lags = OBSERVED_LAGS[OBSERVED_LAGS.lag_minutes >= 0]
coverage_draft = (
    "2026-11-12: AWS NDFD operational 2.5 km CONUS MaxT is available from "
    f"{min(ARCHIVE_DIRECTORY_DAYS)} through the sampled {max(ARCHIVE_DIRECTORY_DAYS)} "
    f"({len(HISTORICAL_DAYS)} historical KNYC sample days plus seven collected days), but "
    f"its {len(STRICT_MISMATCHES)}/{len(STRICT_CONTROL)} unresolved API period comparisons "
    f"and {len(MISMATCHES)}/{len(CONTROL)} unmatched near-window value comparisons do not "
    "establish a drop-in point-in-time archive of forecastGridData; historical Baseline 1 "
    "inputs remain unvalidated pending period/revision alignment or another archive."
)
availability_draft = (
    "2026-11-12: For backfilled forecasts, use the WMO filename issue time and set "
    f"available_at = issue + L, with L = max observed KNYC overlap first-value lag "
    f"({max_lag:.6f} minutes) + one 30-minute poll interval = "
    f"{L.total_seconds() / 60:.6f} minutes; never use S3 Last-Modified. "
    "The invariant requires reconstructed availability no earlier than issue + L and "
    "the first observed matching fetch; this empirical rule is an assumption beyond "
    "the overlap, and repeated values/unequal period encodings limit "
    "publication-time identification."
)
display(Markdown("**Draft coverage finding — not written:**\n\n" + coverage_draft))
display(Markdown("**Draft availability rule — not written:**\n\n" + availability_draft))
SUMMARY = dict(
    input_counts=INPUT_COUNTS.to_dict("records"),
    KNYC_grid_fetches=len(GRIDS["KNYC"]),
    KNYC_grid_issuances=len({r["data"]["properties"]["updateTime"] for r in GRIDS["KNYC"]}),
    KNYC_maximum_vectors=len(
        {
            json.dumps(r["data"]["properties"]["maxTemperature"], sort_keys=True)
            for r in GRIDS["KNYC"]
        }
    ),
    common_comparisons=len(CONTROL),
    common_mismatches=len(MISMATCHES),
    common_no_candidate=int((MISMATCHES.candidates == 0).sum()),
    common_candidate_no_match=int((MISMATCHES.candidates > 0).sum()),
    strict_comparisons=len(STRICT_CONTROL),
    strict_mismatches=len(STRICT_MISMATCHES),
    lag_fields=len(LAGS),
    observed_lags=len(OBSERVED_LAGS),
    positive_lags=len(positive_lags),
    lag_minutes=OBSERVED_LAGS.lag_minutes.describe().to_dict(),
    L_seconds=L.total_seconds(),
    lag_statuses=LAGS.status.value_counts().to_dict(),
    filename_reference_disagreements=int(sum(reference_checks.difference_minutes != 0)),
    filename_reference_files=len(reference_checks),
    proofs=PROOFS,
    estimates=ESTIMATES.to_dict("records"),
    coverage_draft=coverage_draft,
    availability_draft=availability_draft,
    peak_rss_mib=max(r["peak_rss_mib"] for r in DECODE_LOG),
    missing_directory_days=[d.isoformat() for d in MISSING_DIRECTORY_DAYS],
    measured_requests=len({r["url"] for r in REQUEST_LOG}),
    measured_bytes=sum(r["size"] for r in {r["url"]: r for r in REQUEST_LOG}.values()),
    HTTP_requests_this_execution=sum(not r["cached"] for r in REQUEST_LOG),
)
(CACHE / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=str))
print("Saved complete raw-backed reports and summary under", CACHE)
print("HTTP requests this execution:", SUMMARY["HTTP_requests_this_execution"])

Daily MaxT directory census (all domains; not proof of CONUS point values):
First: 2020-04-16 last: 2026-11-13
Missing directories: []
Unsampled CONUS daily inventories: 2310


,first_tested_issue,last_tested_issue,tested_fields
station,,,
KAUS,2020-04-16 15:48:00+00:00,2026-11-13 02:46:00+00:00,6
KDEN,2020-04-16 15:48:00+00:00,2026-11-13 02:46:00+00:00,6
KLAX,2020-04-16 15:48:00+00:00,2026-11-13 02:46:00+00:00,6
KMDW,2020-04-16 15:48:00+00:00,2026-11-13 02:46:00+00:00,6
KMIA,2020-04-16 15:48:00+00:00,2026-11-13 02:46:00+00:00,6
KNYC,2020-04-16 15:48:00+00:00,2026-11-13 02:46:00+00:00,6
KPHL,2020-04-16 15:48:00+00:00,2026-11-13 02:46:00+00:00,6


matching_value_elsewhere_in_overlap
False     97
True     164
Name: fetch/period pairs, dtype: int64

,fetched_at,valid_start,api_f,closest_matching_issue,matching_issue_minus_fetch_minutes,all_values_for_period_in_overlap
0,2026-11-08 07:45:45.303607+00:00,2026-11-11 12:00:00+00:00,61,2026-11-08 11:33:00+00:00,227.244940,"[60, 61, 62, 63, 64]"
1,2026-11-08 07:45:45.303607+00:00,2026-11-12 12:00:00+00:00,57,2026-11-08 11:33:00+00:00,227.244940,"[56, 57, 58, 59, 61]"
2,2026-11-08 07:45:45.303607+00:00,2026-11-13 12:00:00+00:00,49,2026-11-08 11:33:00+00:00,227.244940,"[45, 46, 47, 49, 51, 52, 53]"
3,2026-11-08 07:45:45.303607+00:00,2026-11-15 12:00:00+00:00,55,NaT,NaN,"[49, 51, 52, 53]"
45,2026-11-08 19:45:45.346970+00:00,2026-11-13 12:00:00+00:00,47,2026-11-08 21:33:00+00:00,107.244217,"[45, 46, 47, 49, 51, 52, 53]"
46,2026-11-08 19:45:45.346970+00:00,2026-11-14 12:00:00+00:00,47,2026-11-08 21:33:00+00:00,107.244217,"[46, 47, 49, 51]"
47,2026-11-08 19:45:45.346970+00:00,2026-11-15 12:00:00+00:00,49,2026-11-08 21:33:00+00:00,107.244217,"[49, 51, 52, 53]"
54,2026-11-09 07:45:45.304629+00:00,2026-11-12 12:00:00+00:00,59,2026-11-09 11:33:00+00:00,227.244923,"[56, 57, 58, 59, 61]"
55,2026-11-09 07:45:45.304629+00:00,2026-11-13 12:00:00+00:00,52,2026-11-09 11:33:00+00:00,227.244923,"[45, 46, 47, 49, 51, 52, 53]"
56,2026-11-09 07:45:45.304629+00:00,2026-11-15 12:00:00+00:00,53,2026-11-09 11:33:00+00:00,227.244923,"[49, 51, 52, 53]"


response,fetched_at,api_start,api_end,api_hours,api_f,archive_start,archive_end,archive_f,key,result
2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-07 15:00:00+00:00,2026-11-08 01:00:00+00:00,10.0,63,2026-11-07 12:00:00+00:00,2026-11-08 00:00:00+00:00,63.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,period_not_reproduced
2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-08 12:00:00+00:00,2026-11-09 01:00:00+00:00,13.0,62,2026-11-08 12:00:00+00:00,2026-11-09 00:00:00+00:00,62.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,period_not_reproduced
2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-09 12:00:00+00:00,2026-11-10 01:00:00+00:00,13.0,58,2026-11-09 12:00:00+00:00,2026-11-10 00:00:00+00:00,58.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,period_not_reproduced
2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-10 12:00:00+00:00,2026-11-11 01:00:00+00:00,13.0,61,2026-11-10 12:00:00+00:00,2026-11-11 00:00:00+00:00,61.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072147,period_not_reproduced
2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-11 12:00:00+00:00,2026-11-12 01:00:00+00:00,13.0,63,2026-11-11 12:00:00+00:00,2026-11-12 00:00:00+00:00,63.0,wmo/maxt/2026/11/07/YGUZ97_KWBN_202611072133,period_not_reproduced
2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-12 12:00:00+00:00,2026-11-13 01:00:00+00:00,13.0,58,2026-11-12 12:00:00+00:00,2026-11-13 00:00:00+00:00,58.0,wmo/maxt/2026/11/07/YGUZ97_KWBN_202611072133,period_not_reproduced
2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-13 12:00:00+00:00,2026-11-14 01:00:00+00:00,13.0,51,2026-11-13 12:00:00+00:00,2026-11-14 00:00:00+00:00,51.0,wmo/maxt/2026/11/07/YGUZ97_KWBN_202611072133,period_not_reproduced
2026-11-07T214551.305470Z.json.gz,2026-11-07 21:45:51.305470+00:00,2026-11-14 12:00:00+00:00,2026-11-15 01:00:00+00:00,13.0,51,2026-11-14 12:00:00+00:00,2026-11-15 00:00:00+00:00,51.0,wmo/maxt/2026/11/07/YGUZ97_KWBN_202611072133,period_not_reproduced
2026-11-07T221547.054706Z.json.gz,2026-11-07 22:15:47.054706+00:00,2026-11-07 15:00:00+00:00,2026-11-08 01:00:00+00:00,10.0,63,2026-11-07 12:00:00+00:00,2026-11-08 00:00:00+00:00,63.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072216,period_not_reproduced
2026-11-07T221547.054706Z.json.gz,2026-11-07 22:15:47.054706+00:00,2026-11-08 12:00:00+00:00,2026-11-09 01:00:00+00:00,13.0,62,2026-11-08 12:00:00+00:00,2026-11-09 00:00:00+00:00,62.0,wmo/maxt/2026/11/07/YGUZ98_KWBN_202611072216,period_not_reproduced


response,fetched_at,new_api_vector,valid_start,valid_end,api_f,matched,archive_f,key,candidates,matching_value_elsewhere_in_overlap,closest_matching_issue,closest_matching_key,matching_issue_minus_fetch_minutes,all_values_for_period_in_overlap
2026-11-08T074545.303607Z.json.gz,2026-11-08 07:45:45.303607+00:00,True,2026-11-11 12:00:00+00:00,2026-11-12 00:00:00+00:00,61,False,63.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6,True,2026-11-08 11:33:00+00:00,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611081133,227.244940,"[60, 61, 62, 63, 64]"
2026-11-08T074545.303607Z.json.gz,2026-11-08 07:45:45.303607+00:00,True,2026-11-12 12:00:00+00:00,2026-11-13 00:00:00+00:00,57,False,58.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6,True,2026-11-08 11:33:00+00:00,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611081133,227.244940,"[56, 57, 58, 59, 61]"
2026-11-08T074545.303607Z.json.gz,2026-11-08 07:45:45.303607+00:00,True,2026-11-13 12:00:00+00:00,2026-11-14 00:00:00+00:00,49,False,51.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6,True,2026-11-08 11:33:00+00:00,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611081133,227.244940,"[45, 46, 47, 49, 51, 52, 53]"
2026-11-08T074545.303607Z.json.gz,2026-11-08 07:45:45.303607+00:00,True,2026-11-15 12:00:00+00:00,2026-11-16 00:00:00+00:00,55,False,NaN,NaN,0,False,NaT,NaN,NaN,"[49, 51, 52, 53]"
2026-11-08T081545.319890Z.json.gz,2026-11-08 08:15:45.319890+00:00,False,2026-11-11 12:00:00+00:00,2026-11-12 00:00:00+00:00,61,False,63.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6,True,2026-11-08 11:33:00+00:00,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611081133,197.244668,"[60, 61, 62, 63, 64]"
2026-11-08T081545.319890Z.json.gz,2026-11-08 08:15:45.319890+00:00,False,2026-11-12 12:00:00+00:00,2026-11-13 00:00:00+00:00,57,False,58.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6,True,2026-11-08 11:33:00+00:00,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611081133,197.244668,"[56, 57, 58, 59, 61]"
2026-11-08T081545.319890Z.json.gz,2026-11-08 08:15:45.319890+00:00,False,2026-11-13 12:00:00+00:00,2026-11-14 00:00:00+00:00,49,False,51.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6,True,2026-11-08 11:33:00+00:00,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611081133,197.244668,"[45, 46, 47, 49, 51, 52, 53]"
2026-11-08T081545.319890Z.json.gz,2026-11-08 08:15:45.319890+00:00,False,2026-11-15 12:00:00+00:00,2026-11-16 00:00:00+00:00,55,False,NaN,NaN,0,False,NaT,NaN,NaN,"[49, 51, 52, 53]"
2026-11-08T084545.321347Z.json.gz,2026-11-08 08:45:45.321347+00:00,False,2026-11-11 12:00:00+00:00,2026-11-12 00:00:00+00:00,61,False,63.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6,True,2026-11-08 11:33:00+00:00,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611081133,167.244644,"[60, 61, 62, 63, 64]"
2026-11-08T084545.321347Z.json.gz,2026-11-08 08:45:45.321347+00:00,False,2026-11-12 12:00:00+00:00,2026-11-13 00:00:00+00:00,57,False,58.0,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611080534,6,True,2026-11-08 11:33:00+00:00,wmo/maxt/2026/11/08/YGUZ97_KWBN_202611081133,167.244644,"[56, 57, 58, 59, 61]"


... (7 outputs left out)

REAL DATA FAILURE (retained, not weakened): AWS overlap failed: 2036/2036 exact-period comparisons unproved; 261/1926 common-interval value searches unmatched. No NCEI/MOS fallback was accessed.


**Draft coverage finding — not written:**

2026-11-12: AWS NDFD operational 2.5 km CONUS MaxT is available from 2020-04-16 through the sampled 2026-11-13 (30 historical KNYC sample days plus seven collected days), but its 2036/2036 unresolved API period comparisons and 261/1926 unmatched near-window value comparisons do not establish a drop-in point-in-time archive of forecastGridData; historical Baseline 1 inputs remain unvalidated pending period/revision alignment or another archive.

**Draft availability rule — not written:**

2026-11-12: For backfilled forecasts, use the WMO filename issue time and set available_at = issue + L, with L = max observed KNYC overlap first-value lag (29.756568 minutes) + one 30-minute poll interval = 59.756568 minutes; never use S3 Last-Modified. The invariant requires reconstructed availability no earlier than issue + L and the first observed matching fetch; this empirical rule is an assumption beyond the overlap, and repeated values/unequal period encodings limit publication-time identification.

Saved complete raw-backed reports and summary under /Users/prathik/code/Kalshi-Weather-Bot/data/spike/nws_archive
HTTP requests this execution: 0


## Design choices and limits

- **Explicit GRIB end fields:** a forecast period can start before an issuance. PDT 4.8's
  end timestamp preserves that interval when a convenience lead-time calculation cannot.
- **Containing cell and original polygons:** station identity comes from config and collected
  `/points`; location comes from the collected grid polygon. API grid coordinates are local
  to a WFO and are never reused as CONUS array indices.
- **Separate controls:** a matching temperature over a common interval is useful evidence,
  while unequal wire periods and later-arriving long-range mosaics remain visible. Neither
  is silently reinterpreted as exact point-in-time equivalence.
- **Shared file costs:** one decoded CONUS field supplies all seven stations. Multiplying
  full-file bytes by seven would greatly overstate the backfill.
- **Exact-byte duplicate checks:** comparing decompressed envelopes matches the collector's
  compaction preservation contract, while allowing loose and compacted copies to coexist.

Out of scope, not fixed: resolving API's 13-hour value-support encoding versus NDFD's
12-hour statistical period; deciding the CLI climate-day mapping; establishing publication
lag bounds outside this short overlap; historical WFO domain/polygon changes; complete
non-null point coverage on unsampled days; NCEI/MOS fallback investigation; production
parsers, store accessors, schema changes, and a full historical backfill.